# ICE478 Summer 2026 — Task 1: Data Understanding & Related Work
## Group12_MaleBin · Track 3 (CNN + Attention) · Dataset: **MaleBin**

**Dataset** · *MaleBin: Malware Binary Greyscale Images* — Kaggle
`tashiee/malebin-malware-binary-greyscale-images` (CC BY 4.0).
12,464 grayscale byte-plot images, 39 malware families, 256×256, compiled by the
uploader from **two** sources:

1. **Malimg** (Nataraj et al., *VizSec* 2011) — the 25-family benchmark;
2. a subset of **`walt30/malware-images`**, visualised from MalwareBazaar samples
   following the same byte-to-pixel method.

**Problem type** · single-label multi-class image classification (39 classes).

**Application area** · static malware triage. A PE binary's bytes are read as a
stream, laid out row-wise into a fixed-width 2-D array and rendered as an 8-bit
grayscale image. Different families produce visibly different *textures* because
they share code, packers and resource layouts, which is what makes image
classification work at all here.

---

### What this notebook covers (brief §5, Task 1 — Image EDA A–G, plus H)

| | Item |
|---|---|
| **A** | Summary table — name, source, area, samples, classes, format, resolution, balance, missing/corrupt, duplicates |
| **B** | Class balance — counts, bar chart, majority:minority ratio, and what it does to metrics |
| **C** | Labelled sample grid — which families look alike? |
| **D** | Size / resolution audit |
| **E** | Pixel & texture statistics — mean, median, std, min/max, quartiles, skewness, kurtosis, entropy; histograms, boxplots, violin plots |
| **F** | Image quality — corrupt files, near-constant images, **exact and near-duplicates (the leakage risk)**, correlated features |
| **G** | 2-D structure — correlation heatmap + PCA / t-SNE / UMAP |
| **H** | Interactive Plotly figures |
| — | **Related-work table** (7 papers, 2022–2026) + the research gap |

### How to run this on Kaggle
1. *New Notebook* → **Add Data** → search `MaleBin malware binary greyscale` → **Add**.
2. Settings → **Accelerator: GPU T4/P100** (not needed for Task 1, but keeps all
   five notebooks on one profile), **Internet: off** is fine.
3. *Run All*. Nothing is downloaded; every dependency below ships with the
   Kaggle image.
4. For a ~3-minute wiring check first, set `MALEBIN_FAST=1` in the environment
   cell (or `CFG.fast = True` right after the boot cell).

> ⚠️ **Note on the dataset's own disclaimer.** The uploader states that a newer
> *MaleBin 2.0 RGB* set exists, that resizing in v1 can distort images, and that
> the Malimg half contains **outdated** malware which will not generalise to
> modern threats. We use v1 because it is the version the course dataset list
> points at, and we treat that disclaimer as a stated limitation of the study
> rather than something to hide — it is repeated in every task report.

> ### ⚠️ Before you submit: this notebook must be saved **with its outputs**
>
> The course requires the notebook to show its results. On Kaggle:
>
> 1. **Add Data** → search `MaleBin malware binary greyscale` → **Add**.
> 2. **Settings** → *Accelerator*: **GPU P100 / T4**.
> 3. Set your group number in the boot cell (`CFG.group = "Group00"`).
> 4. **Run All**, then **Save Version → Save & Run All (Commit)**.
> 5. Download that committed version — it contains every table, figure and
>    printed number — and push *that* file to the repo.
>
> For a ~3-minute wiring check first, uncomment `CFG.fast = True` in the boot
> cell. **FAST-mode numbers are meaningless** (64 px, 2 epochs); turn it off
> before producing anything you will quote or submit.

In [ ]:
%%writefile malebin_common.py
"""
malebin_common.py
=================
Shared utilities for the ICE478 (Summer 2026) course project.

Track      : 3 -- CNN + Attention
Dataset    : MaleBin: Malware Binary Greyscale Images
             (kaggle: tashiee/malebin-malware-binary-greyscale-images)
             12,464 grayscale byte-plot images, 39 malware families, 256x256,
             compiled from (a) Malimg (Nataraj et al., 2011) and
                           (b) a MalwareBazaar-derived subset (kaggle: walt30).

This single module is embedded (via %%writefile) at the top of every task
notebook so that each notebook is self-contained and re-runnable on Kaggle
without depending on any other notebook's output.

Design notes that matter for the grading rubric
-----------------------------------------------
* Section 6.2 of the brief requires a *source-based* split, not a random one.
  For byte-plot malware images the analogue of "same subject" is
  "same/near-identical binary": malware families are full of polymorphic
  variants whose byte-plots are near-duplicates.  A random split therefore
  leaks.  `build_dedup_groups()` finds exact duplicates (SHA-1 of raw pixels)
  and near-duplicates (128-bit difference hash, Hamming distance <= T), unions
  them, and every split in this project is a *grouped stratified* split over
  those groups (sklearn StratifiedGroupKFold).  Source-of-origin is also
  recorded and reported.
* Section 6.1 requires the full metric set; `full_metrics()` returns it and
  `macro_f1` is the headline number everywhere (Section 6.2, imbalance rule).
* Section 6.4 requires 5-fold CV (mean +- std) plus a significance test;
  `mcnemar_test`, `wilcoxon_folds` and `friedman_nemenyi` are provided.
"""

from __future__ import annotations

import hashlib
import json
import math
import os
import random
import sys
import time
import warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

__all__ = [
    "CFG", "set_seed", "on_kaggle", "find_dataset_root", "scan_index",
    "MALIMG_25", "normalize_family", "tag_malimg_subset",
    "build_dedup_groups", "grouped_holdout_split", "grouped_kfold",
    "load_images", "ByteImageDataset", "make_loaders",
    "SEBlock", "SpatialAttention", "CBAM", "CoordAtt", "GeM", "MSConvBlock",
    "ByteAttnNet", "build_baseline", "count_params", "model_size_mb",
    "train_model", "predict", "full_metrics", "metrics_frame",
    "plot_confusion", "plot_roc_ovr", "plot_pr_ovr", "plot_history",
    "mcnemar_test", "wilcoxon_folds", "friedman_nemenyi",
    "save_json", "load_json", "banner", "RELATED_WORK",
    "CROSS_FAMILY_GROUPS",
    "SAVED_FIGURES", "_install_headless_hooks",
]

def on_kaggle() -> bool:
    return Path("/kaggle/input").exists()

@dataclass
class Config:
    group: str = "Group00"
    dataset_slug: str = "MaleBin"

    data_root: str | None = os.environ.get("MALEBIN_DATA_ROOT") or None
    out_dir: str = "/kaggle/working" if Path("/kaggle/working").exists() else "./out"

    img_size: int = 224
    cache_size: int = 256
    in_channels: int = 1

    seed: int = 42
    test_frac: float = 0.20
    val_frac: float = 0.15
    n_folds: int = 5
    dhash_threshold: int = 6
    group_by_duplicates: bool = True

    batch_size: int = 64
    epochs: int = 25
    lr: float = 3e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.05
    patience: int = 6
    num_workers: int = 2
    amp: bool = True
    class_weighted_loss: bool = True

    eval_scope: str = "malebin39"

    fast: bool = bool(int(os.environ.get("MALEBIN_FAST", "0")))
    max_per_class: int | None = None

    def __post_init__(self):
        if self.fast:
            self.img_size = 64
            self.cache_size = 64
            self.epochs = 2
            self.n_folds = 2
            self.batch_size = 32
            self.patience = 2
            self.num_workers = 0
            self.amp = False
        for var, attr, cast in (
            ("MALEBIN_IMG_SIZE",   "img_size",   int),
            ("MALEBIN_CACHE_SIZE", "cache_size", int),
            ("MALEBIN_EPOCHS",     "epochs",     int),
            ("MALEBIN_FOLDS",      "n_folds",    int),
            ("MALEBIN_BATCH",      "batch_size", int),
            ("MALEBIN_PATIENCE",   "patience",   int),
            ("MALEBIN_WORKERS",    "num_workers", int),
            ("MALEBIN_LR",         "lr",         float),
            ("MALEBIN_OUT_DIR",    "out_dir",    str),
            ("MALEBIN_EVAL_SCOPE", "eval_scope", str),
            ("MALEBIN_MAX_PER_CLASS", "max_per_class", int),
        ):
            raw = os.environ.get(var)
            if raw not in (None, ""):
                try:
                    setattr(self, attr, cast(raw))
                except ValueError:
                    print(f"[CFG] ignoring {var}={raw!r} (not a valid {cast.__name__})")
        if os.environ.get("MALEBIN_AMP") in ("0", "false", "False"):
            self.amp = False
        Path(self.out_dir).mkdir(parents=True, exist_ok=True)
        for sub in ("models", "figures", "artifacts"):
            (Path(self.out_dir) / sub).mkdir(parents=True, exist_ok=True)

    @property
    def prefix(self) -> str:
        return f"{self.group}_{self.dataset_slug}"

    def path(self, *parts) -> Path:
        return Path(self.out_dir).joinpath(*parts)

    def fig(self, name: str) -> Path:
        return self.path("figures", f"{self.prefix}_{name}")

    def art(self, name: str) -> Path:
        return self.path("artifacts", f"{self.prefix}_{name}")

    def mdl(self, name: str) -> Path:
        return self.path("models", f"{self.prefix}_{name}")

CFG = Config()

def banner(txt: str, ch: str = "=") -> None:
    line = ch * max(70, len(txt) + 4)
    print(f"\n{line}\n  {txt}\n{line}")

def set_seed(seed: int | None = None) -> None:
    seed = CFG.seed if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
    except Exception:
        pass

def save_json(obj, path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=_json_default), encoding="utf-8")
    return path

def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp", ".gif"}

MALIMG_25 = [
    "Adialer.C", "Agent.FYI", "Allaple.A", "Allaple.L", "Alueron.gen!J",
    "Autorun.K", "C2LOP.gen!g", "C2LOP.P", "Dialplatform.B", "Dontovo.A",
    "Fakerean", "Instantaccess", "Lolyda.AA1", "Lolyda.AA2", "Lolyda.AA3",
    "Lolyda.AT", "Malex.gen!J", "Obfuscator.AD", "Rbot!gen", "Skintrim.N",
    "Swizzor.gen!E", "Swizzor.gen!I", "VB.AT", "Wintrim.BX", "Yuner.A",
]

def normalize_family(name: str) -> str:
    """Lower-cased, punctuation-stripped family key for robust matching."""
    return "".join(c for c in str(name).lower() if c.isalnum())

_MALIMG_KEYS = {normalize_family(m) for m in MALIMG_25}

def find_dataset_root(explicit: str | None = None) -> Path:
    """
    Locate the directory whose immediate children are the class folders.

    Works for any of these layouts, which is what makes the notebooks
    portable between Kaggle and a local copy:

        <root>/<class>/*.png
        <root>/MaleBin/<class>/*.png
        <root>/train/<class>/*.png          (also picks up val/ and test/)
    """
    cands: list[Path] = []
    if explicit:
        cands.append(Path(explicit))
    if CFG.data_root:
        cands.append(Path(CFG.data_root))
    cands += [
        Path("/kaggle/input/malebin-malware-binary-greyscale-images"),
        Path("./data/malebin"), Path("./malebin"), Path("./data"),
    ]
    if Path("/kaggle/input").exists():
        cands += sorted(Path("/kaggle/input").iterdir())

    for c in cands:
        if not c.exists() or not c.is_dir():
            continue
        root = _descend_to_class_parent(c)
        if root is not None:
            return root
    raise FileNotFoundError(
        "Could not locate the MaleBin image folders.\n"
        "On Kaggle: Add Data -> search 'MaleBin malware binary greyscale' -> Add.\n"
        "Then set CFG.data_root='/kaggle/input/<slug>' and re-run.\n"
        f"Tried: {[str(c) for c in cands]}"
    )

def _immediate_image_count(d: Path, cap: int = 3) -> int:
    n = 0
    try:
        for p in d.iterdir():
            if p.is_file() and p.suffix.lower() in IMG_EXT:
                n += 1
                if n >= cap:
                    break
    except OSError:
        return 0
    return n

def _descend_to_class_parent(d: Path, depth: int = 0) -> Path | None:
    """Return the shallowest directory that has >=2 image-bearing subfolders."""
    if depth > 4:
        return None
    try:
        subs = sorted(p for p in d.iterdir() if p.is_dir())
    except OSError:
        return None
    with_imgs = [s for s in subs if _immediate_image_count(s) > 0]
    if len(with_imgs) >= 2:
        return d
    for s in subs:
        r = _descend_to_class_parent(s, depth + 1)
        if r is not None:
            return r
    return None

def scan_index(root: Path | None = None, verbose: bool = True) -> pd.DataFrame:
    """
    Walk the dataset and return one row per image.

    Columns
    -------
    path        absolute file path
    family      class folder name (malware family)
    label       integer encoding of `family` (alphabetical, stable)
    n_bytes     file size on disk
    is_malimg   the family is one of the 25 original Malimg families
    """
    root = Path(root) if root is not None else find_dataset_root()
    rows = []
    for fam_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        for p in sorted(fam_dir.rglob("*")):
            if p.is_file() and p.suffix.lower() in IMG_EXT:
                rows.append((str(p), fam_dir.name, p.stat().st_size))
    if not rows:
        for p in sorted(root.rglob("*")):
            if p.is_file() and p.suffix.lower() in IMG_EXT:
                rows.append((str(p), p.parent.name, p.stat().st_size))
    if not rows:
        raise FileNotFoundError(f"No images found under {root}")

    df = pd.DataFrame(rows, columns=["path", "family", "n_bytes"])
    fams = sorted(df["family"].unique())
    df["label"] = df["family"].map({f: i for i, f in enumerate(fams)}).astype(int)
    df["is_malimg"] = df["family"].map(lambda f: normalize_family(f) in _MALIMG_KEYS)
    df = df.sort_values("path", kind="mergesort").reset_index(drop=True)

    if verbose:
        vc = df.family.value_counts()
        banner(f"Dataset index: {root}")
        print(f"  images            : {len(df):,}")
        print(f"  families (classes): {df['family'].nunique()}")
        print(f"  Malimg families   : {df.loc[df.is_malimg,'family'].nunique()}"
              f" / 25  ({df.is_malimg.sum():,} images)")
        print(f"  other families    : {df.loc[~df.is_malimg,'family'].nunique()}"
              f" ({(~df.is_malimg).sum():,} images)")
        print(f"  per-class counts  : min={vc.min()}  max={vc.max()}  "
              f"imbalance ratio={vc.max()/max(vc.min(),1):.2f}")
    return df

def tag_malimg_subset(df: pd.DataFrame, scope: str | None = None) -> pd.DataFrame:
    """
    Restrict + re-encode labels according to CFG.eval_scope.

    'malebin39' -> everything, labels 0..K-1
    'malimg25'  -> only the Malimg families, labels re-encoded 0..24, so the
                   result is directly comparable with published Malimg numbers.
    """
    scope = scope or CFG.eval_scope
    if scope == "malimg25":
        out = df[df["is_malimg"]].copy()
        if out.empty:
            raise ValueError(
                "eval_scope='malimg25' but no folder name matched the 25 Malimg "
                "families. Print df.family.unique() and adjust MALIMG_25.")
    elif scope == "malebin39":
        out = df.copy()
    else:
        raise ValueError(f"unknown eval_scope {scope!r}")
    fams = sorted(out["family"].unique())
    out["label"] = out["family"].map({f: i for i, f in enumerate(fams)}).astype(int)
    out = out.reset_index(drop=True)
    print(f"[scope={scope}] {len(out):,} images, {len(fams)} classes")
    return out

def _load_one(args):
    path, side = args
    from PIL import Image
    with Image.open(path) as im:
        im = im.convert("L")
        if im.size != (side, side):
            im = im.resize((side, side), Image.BILINEAR)
        return np.asarray(im, dtype=np.uint8)

def load_images(df: pd.DataFrame, side: int | None = None,
                workers: int | None = None, verbose: bool = True) -> np.ndarray:
    """
    Decode every image once into a single uint8 array  (N, side, side).

    12,464 x 256 x 256 uint8 == 817 MB, which fits comfortably in Kaggle RAM and
    removes disk I/O from the training loop entirely (a large speed win over an
    ImageFolder pipeline that re-decodes a PNG on every epoch).
    """
    side = side or CFG.cache_size
    n = len(df)
    out = np.zeros((n, side, side), dtype=np.uint8)
    paths = df["path"].tolist()
    t0 = time.time()

    if workers is None:
        workers = 0 if (CFG.fast or n < 400) else min(8, (os.cpu_count() or 2))

    done = False
    if workers and workers > 1:
        from concurrent.futures import ThreadPoolExecutor
        try:
            with ThreadPoolExecutor(max_workers=workers) as ex:
                for i, arr in enumerate(ex.map(_load_one,
                                               ((p, side) for p in paths),
                                               chunksize=64)):
                    out[i] = arr
            done = True
        except Exception as e:
            print(f"  [load_images] parallel decode failed ({e!r}); serial fallback")
    if not done:
        try:
            from tqdm.auto import tqdm
            it = tqdm(paths, desc=f"decode@{side}", unit="img")
        except Exception:
            it = paths
        for i, p in enumerate(it):
            out[i] = _load_one((p, side))

    if verbose:
        print(f"  cached {n:,} images at {side}x{side} "
              f"({out.nbytes/1e6:.0f} MB) in {time.time()-t0:.1f}s")
    return out

_POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def _dhash128(imgs: np.ndarray) -> np.ndarray:
    """
    128-bit difference hash per image, packed into (N, 16) uint8:
    64 bits from horizontal gradients + 64 bits from vertical gradients.
    Pure numpy, so there is no `imagehash` dependency to install on Kaggle.
    """
    n, side = imgs.shape[0], imgs.shape[1]
    idx = np.linspace(0, side, 10).astype(int)
    small = np.empty((n, 9, 9), dtype=np.float32)
    for r in range(9):
        r0, r1 = idx[r], max(idx[r] + 1, idx[r + 1])
        for c in range(9):
            c0, c1 = idx[c], max(idx[c] + 1, idx[c + 1])
            small[:, r, c] = imgs[:, r0:r1, c0:c1].reshape(n, -1).mean(axis=1)
    hbits = (small[:, :8, :8] > small[:, :8, 1:9]).reshape(n, 64)
    vbits = (small[:, :8, :8] > small[:, 1:9, :8]).reshape(n, 64)
    return np.packbits(np.concatenate([hbits, vbits], axis=1), axis=1)

CROSS_FAMILY_GROUPS: list[dict] = []

def build_dedup_groups(imgs: np.ndarray, df: pd.DataFrame,
                       threshold: int | None = None,
                       verbose: bool = True) -> np.ndarray:
    """
    Return an integer group id per image so that exact/near duplicates share an
    id.  Near-duplicate search is run *within* each family, so a group can never
    span two families (a cross-family collision would be a labelling issue, not
    a duplicate); we still assert that and report it.
    """
    threshold = CFG.dhash_threshold if threshold is None else threshold
    n = len(df)
    parent = np.arange(n)

    def find(a: int) -> int:
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return int(a)

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)

    if not CFG.group_by_duplicates:
        if verbose:
            print("  [!] CFG.group_by_duplicates=False -> every image is its own "
                  "group (leakage-demonstration mode only)")
        return np.arange(n)

    first: dict[str, int] = {}
    n_exact = 0
    for i in range(n):
        h = hashlib.sha1(imgs[i].tobytes()).hexdigest()
        if h in first:
            union(first[h], i)
            n_exact += 1
        else:
            first[h] = i

    hashes = _dhash128(imgs)
    labels = df["label"].to_numpy()
    n_near = 0
    for lab in np.unique(labels):
        idx = np.flatnonzero(labels == lab)
        if len(idx) < 2:
            continue
        H = hashes[idx]
        B = 512
        for s in range(0, len(idx), B):
            blk = H[s:s + B]
            d = _POPCOUNT[np.bitwise_xor(blk[:, None, :], H[None, :, :])].sum(-1)
            ii, jj = np.nonzero(d <= threshold)
            for a, b in zip(ii, jj):
                gi, gj = int(idx[s + a]), int(idx[b])
                if gi < gj:
                    union(gi, gj)
                    n_near += 1

    raw = np.array([find(i) for i in range(n)])
    _, groups = np.unique(raw, return_inverse=True)

    gf = pd.DataFrame({"g": groups, "f": df["family"].to_numpy()})
    spanning = gf.groupby("g")["f"].nunique() > 1
    cross = int(spanning.sum())

    CROSS_FAMILY_GROUPS.clear()
    if cross:
        bad = spanning[spanning].index.to_numpy()
        fam = df["family"].to_numpy()
        for g in bad:
            members = np.flatnonzero(groups == g)
            CROSS_FAMILY_GROUPS.append(
                {"group": int(g), "n_images": int(len(members)),
                 "families": sorted(set(fam[members].tolist()))})
        pairs = pd.Series(
            [" + ".join(d["families"]) for d in CROSS_FAMILY_GROUPS]
        ).value_counts()
        print(f"  [!] {cross:,} duplicate group(s) span more than one family -- "
              f"these are byte-identical images filed under two labels.")
        for label, k in pairs.items():
            print(f"      {label}: {k:,} group(s)")
        print("      They are kept in ONE group, so no split can put an image "
              "in train and its identical twin in test.")
        print("      NOTE: two families that share images are not separable by "
              "any classifier; this caps the achievable macro-F1.")

    if verbose:
        sizes = pd.Series(groups).value_counts()
        n_groups = int(groups.max()) + 1
        print(f"  exact-duplicate links     : {n_exact:,}")
        print(f"  near-duplicate links      : {n_near:,} "
              f"(dHash Hamming <= {threshold}/128)")
        print(f"  duplicate groups          : {n_groups:,} for {n:,} images "
              f"({100*(1-n_groups/n):.1f}% collapse)")
        print(f"  largest group             : {sizes.max()} images")
        print(f"  groups spanning >1 family : {cross}   (dataset labelling defect)")
    return groups

def grouped_holdout_split(df: pd.DataFrame, groups: np.ndarray,
                          test_frac: float | None = None,
                          val_frac: float | None = None,
                          seed: int | None = None,
                          verbose: bool = True):
    """
    Grouped + stratified train / val / test indices.

    StratifiedGroupKFold is applied twice so that (a) no duplicate group is
    split across subsets and (b) every family keeps roughly its global share.
    """
    from sklearn.model_selection import StratifiedGroupKFold
    test_frac = CFG.test_frac if test_frac is None else test_frac
    val_frac = CFG.val_frac if val_frac is None else val_frac
    seed = CFG.seed if seed is None else seed
    y = df["label"].to_numpy()

    k_test = max(2, int(round(1.0 / test_frac)))
    sgkf = StratifiedGroupKFold(n_splits=k_test, shuffle=True, random_state=seed)
    pool_idx, test_idx = next(sgkf.split(np.zeros(len(y)), y, groups))

    k_val = max(2, int(round(1.0 / val_frac)))
    sgkf2 = StratifiedGroupKFold(n_splits=k_val, shuffle=True, random_state=seed + 1)
    sub_tr, sub_va = next(sgkf2.split(np.zeros(len(pool_idx)),
                                      y[pool_idx], groups[pool_idx]))
    train_idx, val_idx = pool_idx[sub_tr], pool_idx[sub_va]

    assert_no_group_leak({"train": train_idx, "val": val_idx, "test": test_idx},
                         groups)
    if verbose:
        print(f"  train {len(train_idx):,} | val {len(val_idx):,} "
              f"| test {len(test_idx):,}")
        print(f"  classes present -> train {len(set(y[train_idx]))}, "
              f"val {len(set(y[val_idx]))}, test {len(set(y[test_idx]))} "
              f"(dataset has {len(set(y))})")
        print("  leakage check: no duplicate group appears in two subsets  [OK]")
    return train_idx, val_idx, test_idx

def grouped_kfold(df: pd.DataFrame, groups: np.ndarray,
                  n_folds: int | None = None, seed: int | None = None):
    """5-fold *stratified grouped* CV folds (brief Sec. 6.4)."""
    from sklearn.model_selection import StratifiedGroupKFold
    n_folds = CFG.n_folds if n_folds is None else n_folds
    seed = CFG.seed if seed is None else seed
    y = df["label"].to_numpy()
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = list(sgkf.split(np.zeros(len(y)), y, groups))
    for k, (tr, te) in enumerate(folds):
        assert not (set(groups[tr]) & set(groups[te])), f"group leak in fold {k}"
    return folds

def assert_no_group_leak(subsets: dict, groups: np.ndarray) -> None:
    names = list(subsets)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a = set(groups[subsets[names[i]]])
            b = set(groups[subsets[names[j]]])
            inter = a & b
            if inter:
                raise AssertionError(
                    f"LEAKAGE: {len(inter)} duplicate group(s) shared between "
                    f"'{names[i]}' and '{names[j]}'")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ByteImageDataset(Dataset):
    """
    Serves cached uint8 byte-plots as normalised float tensors.

    Augmentation policy -- a design decision we defend in the report
    ---------------------------------------------------------------
    A byte-plot is NOT a natural image.  A pixel's (row, col) position is the
    byte's offset in the file: row r covers bytes [r*W, (r+1)*W).  Therefore
      * horizontal / vertical flips are meaningless -- they reverse the byte
        order of a PE file, producing an input that can never occur;
      * rotations are meaningless for the same reason;
      * brightness / contrast jitter rewrites byte *values* -- also invalid.
    What legitimately varies between two variants of the same family is
      * where a section starts (inserted junk, padding)  -> vertical roll
      * the file being longer or shorter                 -> row-crop / scale
      * a region being packed, encrypted or overwritten  -> random erasing
    `aug='byte'` implements exactly those three.  `aug='naive'` is the usual
    flip+rotate recipe and exists only so the Task-3 ablation can measure it.
    """

    def __init__(self, imgs, labels, indices, out_size=None, aug="none",
                 in_channels=None):
        assert aug in ("none", "byte", "naive"), aug
        self.imgs = imgs
        self.labels = np.asarray(labels)
        self.indices = np.asarray(indices)
        self.out_size = int(out_size or CFG.img_size)
        self.aug = aug
        self.in_channels = int(in_channels or CFG.in_channels)
        self._epoch = 0

    def __len__(self):
        return len(self.indices)

    def set_epoch(self, e):
        self._epoch = int(e)

    @staticmethod
    def _aug_byte(a, rng):
        s = a.shape[0]
        if rng.random() < 0.7:
            a = np.roll(a, int(rng.integers(-s // 8, s // 8 + 1)), axis=0)
        if rng.random() < 0.5:
            keep = max(4, int(s * rng.uniform(0.85, 1.0)))
            top = int(rng.integers(0, s - keep + 1))
            a = a[top:top + keep, :]
        if rng.random() < 0.35:
            h = int(rng.integers(max(1, a.shape[0] // 16), max(2, a.shape[0] // 5)))
            w = int(rng.integers(max(1, a.shape[1] // 8), max(2, a.shape[1])))
            r0 = int(rng.integers(0, a.shape[0] - h + 1))
            c0 = int(rng.integers(0, a.shape[1] - w + 1))
            fill = int(rng.choice([0, 255, int(rng.integers(0, 256))]))
            a = a.copy()
            a[r0:r0 + h, c0:c0 + w] = fill
        return a

    @staticmethod
    def _aug_naive(a, rng):
        if rng.random() < 0.5:
            a = a[:, ::-1]
        if rng.random() < 0.5:
            a = a[::-1, :]
        k = int(rng.integers(0, 4))
        if k:
            a = np.rot90(a, k)
        return np.ascontiguousarray(a)

    def __getitem__(self, i):
        gi = int(self.indices[i])
        a = self.imgs[gi]
        if self.aug != "none":
            rng = np.random.default_rng(
                (CFG.seed * 1_000_003 + gi * 7919 + self._epoch * 104_729
                 + int(torch.randint(0, 1 << 30, (1,)).item())) % (2 ** 63))
            a = self._aug_byte(a, rng) if self.aug == "byte" else self._aug_naive(a, rng)
        t = torch.from_numpy(np.ascontiguousarray(a)).float().div_(255.0)[None]
        if t.shape[-1] != self.out_size or t.shape[-2] != self.out_size:
            t = F.interpolate(t[None], size=(self.out_size, self.out_size),
                              mode="bilinear", align_corners=False)[0]
        t = (t - t.mean()) / (t.std() + 1e-5)
        if self.in_channels == 3:
            t = t.repeat(3, 1, 1)
        return t, int(self.labels[gi])

def make_loaders(imgs, df, train_idx, val_idx, test_idx=None,
                 batch_size=None, aug="byte", in_channels=None, out_size=None):
    bs = int(batch_size or CFG.batch_size)
    y = df["label"].to_numpy()
    kw = dict(out_size=out_size, in_channels=in_channels)
    common = dict(num_workers=CFG.num_workers,
                  pin_memory=torch.cuda.is_available(),
                  persistent_workers=CFG.num_workers > 0)
    tr = DataLoader(ByteImageDataset(imgs, y, train_idx, aug=aug, **kw),
                    batch_size=bs, shuffle=True, drop_last=False, **common)
    va = DataLoader(ByteImageDataset(imgs, y, val_idx, aug="none", **kw),
                    batch_size=bs * 2, shuffle=False, **common)
    te = None
    if test_idx is not None:
        te = DataLoader(ByteImageDataset(imgs, y, test_idx, aug="none", **kw),
                        batch_size=bs * 2, shuffle=False, **common)
    return tr, va, te

class SEBlock(nn.Module):
    """Channel attention -- 'which feature maps matter for this file?'"""

    def __init__(self, ch, r=8):
        super().__init__()
        hid = max(4, ch // r)
        self.fc1 = nn.Conv2d(ch, hid, 1)
        self.fc2 = nn.Conv2d(hid, ch, 1)

    def forward(self, x):
        s = F.adaptive_avg_pool2d(x, 1)
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(s), inplace=True)))
        return x * s

class SpatialAttention(nn.Module):
    """
    Spatial attention -- 'which byte offsets matter?'
    Descriptor = [channel-max, channel-mean] -> 7x7 conv -> sigmoid mask.
    """

    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)

    def forward(self, x):
        mx = x.max(dim=1, keepdim=True)[0]
        av = x.mean(dim=1, keepdim=True)
        self.last_map = torch.sigmoid(self.conv(torch.cat([mx, av], dim=1)))
        return x * self.last_map

class CBAM(nn.Module):
    """CBAM = channel attention (avg+max squeeze) then spatial attention."""

    def __init__(self, ch, r=8, k=7):
        super().__init__()
        hid = max(4, ch // r)
        self.mlp = nn.Sequential(nn.Conv2d(ch, hid, 1), nn.ReLU(inplace=True),
                                 nn.Conv2d(hid, ch, 1))
        self.sa = SpatialAttention(k)

    def forward(self, x):
        c = self.mlp(F.adaptive_avg_pool2d(x, 1)) + self.mlp(F.adaptive_max_pool2d(x, 1))
        self.last_channel = torch.sigmoid(c)
        return self.sa(x * self.last_channel)

class CoordAtt(nn.Module):
    """
    Coordinate attention (Hou et al., CVPR 2021).

    Why this is the right attention for byte-plots
    ----------------------------------------------
    Pooling happens along one axis at a time, so the module emits one gate per
    ROW and one per COLUMN and never destroys position.  In a byte-plot the row
    index *is* the byte offset: the PE header sits at the top, .text below it,
    .data / .rsrc lower, zero padding at the bottom.  A per-row gate can
    therefore express "attend to the import-table region", a structurally
    meaningful statement that globally-pooled channel attention cannot make.
    `last_h` / `last_w` are kept for the Task-3 explainability notebook.
    """

    def __init__(self, ch, r=16):
        super().__init__()
        hid = max(8, ch // r)
        self.conv1 = nn.Conv2d(ch, hid, 1)
        self.bn1 = nn.BatchNorm2d(hid)
        self.conv_h = nn.Conv2d(hid, ch, 1)
        self.conv_w = nn.Conv2d(hid, ch, 1)

    def forward(self, x):
        n, c, h, w = x.shape
        xh = x.mean(dim=3, keepdim=True)
        xw = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y = F.hardswish(self.bn1(self.conv1(torch.cat([xh, xw], dim=2))))
        yh, yw = torch.split(y, [h, w], dim=2)
        ah = torch.sigmoid(self.conv_h(yh))
        aw = torch.sigmoid(self.conv_w(yw).permute(0, 1, 3, 2))
        self.last_h, self.last_w = ah.detach(), aw.detach()
        return x * ah * aw

def attention_module(kind, ch):
    kind = (kind or "none").lower()
    if kind in ("none", ""):
        return nn.Identity()
    if kind == "se":
        return SEBlock(ch)
    if kind == "spatial":
        return SpatialAttention()
    if kind == "cbam":
        return CBAM(ch)
    if kind == "coord":
        return CoordAtt(ch)
    if kind in ("cbam+coord", "full"):
        return nn.Sequential(CBAM(ch), CoordAtt(ch))
    raise ValueError(f"unknown attention {kind!r}")

class GeM(nn.Module):
    """
    Generalised-mean pooling  (mean(x**p))**(1/p)  with learnable p.
    p=1 is average pooling, p->inf is max pooling.  Byte-plot families are
    distinguished by *texture energy*, so a learnable p lets the head settle
    anywhere between "average texture" and "the single most distinctive block"
    instead of us guessing which one is right.
    """

    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p)))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0, max=8.0)
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(p), 1) \
                .pow(1.0 / p).flatten(1)

class GapGmp(nn.Module):
    def forward(self, x):
        return torch.cat([F.adaptive_avg_pool2d(x, 1).flatten(1),
                          F.adaptive_max_pool2d(x, 1).flatten(1)], dim=1)

def pooling_module(kind, ch):
    kind = (kind or "gem").lower()
    if kind == "gem":
        return GeM(), ch
    if kind == "gap":
        return nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten()), ch
    if kind == "gap+gmp":
        return GapGmp(), ch * 2
    raise ValueError(f"unknown pool {kind!r}")

class MSConvBlock(nn.Module):
    """
    Multi-scale residual block: three parallel views, concatenated, fused 1x1.
      * 3x3               -> fine byte texture (opcode-level n-grams)
      * 3x3 dilation 2    -> 5x5 receptive field at 3x3 cost (basic-block level)
      * 3x3 dilation 3    -> 7x7 receptive field           (section level)
    A family's signature lives at several scales at once (byte patterns inside a
    function, repeated function bodies, whole padded sections), so one kernel
    size must compromise.  Same motivation as the multi-scale-kernel blocks of
    PAFE and IMCMK-CNN, but the dilated form keeps a plain 3x3 parameter count.
    Set multiscale=False for the ablation.
    """

    def __init__(self, cin, cout, use_bn=True, multiscale=True):
        super().__init__()
        self.multiscale = multiscale
        norm = (lambda c: nn.BatchNorm2d(c)) if use_bn else (lambda c: nn.Identity())
        if multiscale:
            br = max(8, cout // 3)
            self.b1 = nn.Sequential(nn.Conv2d(cin, br, 3, padding=1, bias=False),
                                    norm(br), nn.ReLU(inplace=True))
            self.b2 = nn.Sequential(nn.Conv2d(cin, br, 3, padding=2, dilation=2,
                                              bias=False), norm(br), nn.ReLU(inplace=True))
            self.b3 = nn.Sequential(nn.Conv2d(cin, br, 3, padding=3, dilation=3,
                                              bias=False), norm(br), nn.ReLU(inplace=True))
            self.fuse = nn.Sequential(nn.Conv2d(br * 3, cout, 1, bias=False), norm(cout))
        else:
            self.b1 = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                                    norm(cout), nn.ReLU(inplace=True))
            self.fuse = nn.Sequential(nn.Conv2d(cout, cout, 3, padding=1, bias=False),
                                      norm(cout))
        self.short = (nn.Identity() if cin == cout else
                      nn.Sequential(nn.Conv2d(cin, cout, 1, bias=False), norm(cout)))
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        h = (torch.cat([self.b1(x), self.b2(x), self.b3(x)], dim=1)
             if self.multiscale else self.b1(x))
        return self.act(self.fuse(h) + self.short(x))

class ByteAttnNet(nn.Module):
    """
    ByteAttnNet: a from-scratch CNN with a two-part attention stack, designed
    specifically for malware byte-plot images.

    Forward path
    ------------
        input  (B, 1, S, S)   byte-plot, per-image standardised
          |
        STEM      3x3 conv stride 2 -> BN -> ReLU            S -> S/2
          |
        STAGE i in 1..4:
            MSConvBlock(c_{i-1} -> c_i)     multi-scale texture, residual
            [MSConvBlock(c_i -> c_i)] x (depth_i - 1)
            attention(c_i)                  CBAM then CoordAtt
            MaxPool 2x2  (stages 1..3 only)
          |
        POOL      GeM (learnable p)                      -> (B, c_4)
          |
        HEAD      Dropout -> Linear(c_4 -> n_classes)
    Channels default to (48, 96, 192, 320); depth defaults to (1, 1, 2, 2).

    Why it should beat the baselines
    --------------------------------
    1. The ImageNet-pretrained baselines carry a *natural-image* prior (oriented
       edges, colour-opponent blobs, object parts).  A byte-plot has none of
       those; what it has is stationary texture whose statistics change with
       byte offset.  Fine-tuning has to unlearn the prior first, and it does so
       with 23M+ parameters on ~10k images.
    2. CBAM re-weights *channels* (which texture detectors this family needs)
       and then *space* (which regions carry them).
    3. CoordAtt adds the piece that matters most here: it factorises attention
       into a per-ROW and a per-COLUMN gate, and the row index of a byte-plot is
       literally the byte offset in the file.  The network can therefore learn
       "for family X, the informative bytes sit ~15-25% into the file", which no
       flatten-then-pool channel attention can represent.
    4. Multi-scale dilated branches see opcode-level, function-level and
       section-level structure simultaneously.
    5. GeM lets the readout interpolate between average and max texture energy.
    6. It is ~1-3M parameters, i.e. ~10x smaller than ResNet50, which matters
       for the training-time / model-size columns the brief asks for.

    Every design choice above is switchable, so the Task-3 ablation can measure
    each one instead of us asserting it.
    """

    def __init__(self, n_classes, in_ch=1, channels=(48, 96, 192, 320),
                 depth=(1, 1, 2, 2), attention="cbam+coord", pool="gem",
                 dropout=0.3, use_bn=True, multiscale=True, stem_stride=2):
        super().__init__()
        self.hparams = dict(n_classes=n_classes, in_ch=in_ch,
                            channels=tuple(channels), depth=tuple(depth),
                            attention=attention, pool=pool, dropout=dropout,
                            use_bn=use_bn, multiscale=multiscale,
                            stem_stride=stem_stride)
        norm = (lambda c: nn.BatchNorm2d(c)) if use_bn else (lambda c: nn.Identity())
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, channels[0], 3, stride=stem_stride, padding=1, bias=False),
            norm(channels[0]), nn.ReLU(inplace=True))

        stages, cin = [], channels[0]
        for i, (cout, d) in enumerate(zip(channels, depth)):
            blocks = [MSConvBlock(cin, cout, use_bn=use_bn, multiscale=multiscale)]
            blocks += [MSConvBlock(cout, cout, use_bn=use_bn, multiscale=multiscale)
                       for _ in range(max(0, d - 1))]
            blocks.append(attention_module(attention, cout))
            if i < len(channels) - 1:
                blocks.append(nn.MaxPool2d(2))
            stages.append(nn.Sequential(*blocks))
            cin = cout
        self.stages = nn.ModuleList(stages)
        self.pool, feat = pooling_module(pool, cin)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(feat, n_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def features(self, x):
        """Last conv feature map -- the Grad-CAM target."""
        x = self.stem(x)
        for s in self.stages:
            x = s(x)
        return x

    def forward(self, x):
        return self.fc(self.drop(self.pool(self.features(x))))

    @property
    def gradcam_layer(self):
        """Deepest spatial tensor producer -> what Grad-CAM should hook."""
        return self.stages[-1]

BASELINE_NAMES = ["SimpleCNN", "ResNet50", "DenseNet121", "MobileNetV3-Small",
                  "EfficientNet-B0", "VGG16"]

class SimpleCNN(nn.Module):
    """The textbook 4-block VGG-style CNN, trained from scratch, no attention."""

    def __init__(self, n_classes, in_ch=1, width=32, dropout=0.3):
        super().__init__()
        chs = [width, width * 2, width * 4, width * 8]
        layers, c = [], in_ch
        for ch in chs:
            layers += [nn.Conv2d(c, ch, 3, padding=1, bias=False),
                       nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
                       nn.Conv2d(ch, ch, 3, padding=1, bias=False),
                       nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
                       nn.MaxPool2d(2)]
            c = ch
        self.body = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(dropout), nn.Linear(c, n_classes))

    def features(self, x):
        return self.body(x)

    def forward(self, x):
        return self.head(self.body(x))

    @property
    def gradcam_layer(self):
        return self.body

def build_baseline(name, n_classes, pretrained=True, in_ch=3, dropout=0.3):
    """
    Return (model, in_channels_required).

    Pretrained torchvision backbones expect 3 channels, so `make_loaders` is
    called with in_channels=3 for them and the grayscale plane is repeated.
    This is the standard protocol in the malware-image literature (PAFE,
    IMCEC, Alshomrani et al.) so the comparison stays fair.
    """
    name = name.strip()
    if name == "SimpleCNN":
        return SimpleCNN(n_classes, in_ch=1, dropout=dropout), 1

    import torchvision.models as tvm
    W = "DEFAULT" if pretrained else None

    if name == "ResNet50":
        m = tvm.resnet50(weights=W)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
    elif name == "ResNet18":
        m = tvm.resnet18(weights=W)
        m.fc = nn.Linear(m.fc.in_features, n_classes)
    elif name == "DenseNet121":
        m = tvm.densenet121(weights=W)
        m.classifier = nn.Linear(m.classifier.in_features, n_classes)
    elif name in ("MobileNetV3-Small", "MobileNetV3"):
        m = tvm.mobilenet_v3_small(weights=W)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n_classes)
    elif name == "MobileNetV2":
        m = tvm.mobilenet_v2(weights=W)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n_classes)
    elif name in ("EfficientNet-B0", "EfficientNetB0"):
        m = tvm.efficientnet_b0(weights=W)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n_classes)
    elif name == "VGG16":
        m = tvm.vgg16_bn(weights=W)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n_classes)
    else:
        raise ValueError(f"unknown baseline {name!r}; choose from {BASELINE_NAMES}")
    return m, 3

def count_params(model, trainable_only=True):
    ps = model.parameters()
    return int(sum(p.numel() for p in ps if (p.requires_grad or not trainable_only)))

def model_size_mb(model):
    """On-disk float32 size of the state dict, in MB."""
    n = sum(p.numel() for p in model.parameters())
    b = sum(buf.numel() for buf in model.buffers())
    return (n + b) * 4 / 1e6

def gradcam_target(model):
    """Best-effort last-conv-stage locator, for any of the models above."""
    if hasattr(model, "gradcam_layer"):
        return model.gradcam_layer
    for attr in ("layer4", "features", "body"):
        if hasattr(model, attr):
            return getattr(model, attr)
    convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    if not convs:
        raise RuntimeError("no Conv2d found for Grad-CAM")
    return convs[-1]

def class_weights_from(labels, n_classes):
    """
    Inverse-frequency weights, normalised to mean 1.
    Used because the brief (Sec. 6.2) grades macro-F1 and per-class recall:
    an unweighted loss on an imbalanced set optimises the majority classes.
    """
    cnt = np.bincount(np.asarray(labels), minlength=n_classes).astype(np.float64)
    cnt[cnt == 0] = 1.0
    w = cnt.sum() / (n_classes * cnt)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)

@torch.no_grad()
def predict(model, loader, device=None, return_logits=False):
    """Return (y_true, y_pred, y_prob) over a loader, in loader order."""
    device = device or DEVICE
    model.eval().to(device)
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type,
                            enabled=(CFG.amp and device.type == "cuda")):
            out = model(xb)
        ps.append(out.float().cpu())
        ys.append(yb)
    logits = torch.cat(ps)
    y_true = torch.cat(ys).numpy()
    prob = torch.softmax(logits, dim=1).numpy()
    y_pred = prob.argmax(1)
    if return_logits:
        return y_true, y_pred, prob, logits.numpy()
    return y_true, y_pred, prob

def train_model(model, train_loader, val_loader, n_classes, *,
                epochs=None, lr=None, weight_decay=None, patience=None,
                label_smoothing=None, class_weighted=None, device=None,
                scheduler="cosine", optimizer="adamw", tag="model",
                verbose=True, save_path=None):
    """
    One generic training loop used by every model in the project, so that the
    comparison between baselines and the proposed model is apples-to-apples
    (same optimiser family, same schedule, same early-stopping criterion, same
    class weighting).  Selection metric is *validation macro-F1*, never accuracy.

    Returns (best_state_dict, history_dataframe, summary_dict).
    """
    from sklearn.metrics import f1_score
    device = device or DEVICE
    epochs = int(epochs or CFG.epochs)
    lr = float(lr if lr is not None else CFG.lr)
    weight_decay = float(weight_decay if weight_decay is not None else CFG.weight_decay)
    patience = int(patience or CFG.patience)
    ls = float(label_smoothing if label_smoothing is not None else CFG.label_smoothing)
    cw_on = CFG.class_weighted_loss if class_weighted is None else class_weighted

    model = model.to(device)
    ytr = np.concatenate([train_loader.dataset.labels[train_loader.dataset.indices]])
    weight = class_weights_from(ytr, n_classes).to(device) if cw_on else None
    crit = nn.CrossEntropyLoss(weight=weight, label_smoothing=ls)

    if optimizer.lower() == "adamw":
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer.lower() == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer.lower() == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                              weight_decay=weight_decay, nesterov=True)
    else:
        raise ValueError(optimizer)

    steps = max(1, len(train_loader))
    if scheduler == "cosine":
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=lr, total_steps=epochs * steps, pct_start=0.25)
        per_step = True
    elif scheduler == "plateau":
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="max", factor=0.5, patience=2)
        per_step = False
    elif scheduler == "step":
        sched = torch.optim.lr_scheduler.StepLR(opt, step_size=max(1, epochs // 3),
                                                gamma=0.3)
        per_step = False
    else:
        sched, per_step = None, False

    use_amp = bool(CFG.amp and device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_f1, best_state, best_ep, bad = -1.0, None, -1, 0
    hist, t_start = [], time.time()

    for ep in range(1, epochs + 1):
        if hasattr(train_loader.dataset, "set_epoch"):
            train_loader.dataset.set_epoch(ep)
        model.train()
        run_loss, seen, correct = 0.0, 0, 0
        t0 = time.time()
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(xb)
                loss = crit(out, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt)
            scaler.update()
            if sched is not None and per_step:
                sched.step()
            run_loss += loss.item() * yb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            seen += yb.size(0)

        yv, pv, _ = predict(model, val_loader, device)
        vf1 = f1_score(yv, pv, average="macro", zero_division=0)
        vacc = float((yv == pv).mean())
        if sched is not None and not per_step:
            sched.step(vf1) if scheduler == "plateau" else sched.step()

        hist.append(dict(epoch=ep, train_loss=run_loss / max(1, seen),
                         train_acc=correct / max(1, seen),
                         val_acc=vacc, val_macro_f1=vf1,
                         lr=opt.param_groups[0]["lr"], secs=time.time() - t0))
        if verbose:
            print(f"  [{tag}] ep {ep:>3}/{epochs}  loss {hist[-1]['train_loss']:.4f}"
                  f"  tr_acc {hist[-1]['train_acc']:.4f}"
                  f"  val_acc {vacc:.4f}  val_macroF1 {vf1:.4f}"
                  f"  ({hist[-1]['secs']:.1f}s)")

        if vf1 > best_f1 + 1e-5:
            best_f1, best_ep, bad = vf1, ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                if verbose:
                    print(f"  [{tag}] early stop at epoch {ep} "
                          f"(best epoch {best_ep}, val macro-F1 {best_f1:.4f})")
                break

    train_secs = time.time() - t_start
    if best_state is not None:
        model.load_state_dict(best_state)
    summary = dict(tag=tag, best_epoch=best_ep, best_val_macro_f1=float(best_f1),
                   epochs_run=len(hist), train_seconds=float(train_secs),
                   params=count_params(model), size_mb=model_size_mb(model),
                   device=str(device))
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        torch.save({"state_dict": best_state,
                    "hparams": getattr(model, "hparams", {}),
                    "summary": summary}, save_path)
        if verbose:
            print(f"  [{tag}] checkpoint -> {save_path}")
    return best_state, pd.DataFrame(hist), summary

def full_metrics(y_true, y_pred, y_prob=None, class_names=None, name="model"):
    """
    Everything Sec. 6.1 asks for, in one dict:
      accuracy, balanced accuracy, macro/weighted precision-recall-F1,
      Cohen's kappa, MCC, macro & weighted one-vs-rest ROC-AUC, macro average
      precision, per-class report, confusion matrix.
    macro_f1 is the headline metric for this project (imbalanced data).
    """
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                                 precision_recall_fscore_support, cohen_kappa_score,
                                 matthews_corrcoef, roc_auc_score,
                                 average_precision_score, classification_report,
                                 confusion_matrix)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    labels = np.arange(len(class_names)) if class_names is not None else \
        np.unique(np.concatenate([y_true, y_pred]))

    pm, rm, fm, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0, labels=labels)
    pw, rw, fw, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0, labels=labels)

    out = dict(
        model=name,
        accuracy=float(accuracy_score(y_true, y_pred)),
        balanced_accuracy=float(balanced_accuracy_score(y_true, y_pred)),
        precision_macro=float(pm), recall_macro=float(rm), f1_macro=float(fm),
        precision_weighted=float(pw), recall_weighted=float(rw), f1_weighted=float(fw),
        cohen_kappa=float(cohen_kappa_score(y_true, y_pred)),
        mcc=float(matthews_corrcoef(y_true, y_pred)),
        n_test=int(len(y_true)), n_classes=int(len(labels)),
    )
    if y_prob is not None:
        y_prob = np.asarray(y_prob)
        Y = np.zeros((len(y_true), y_prob.shape[1]), dtype=np.int8)
        Y[np.arange(len(y_true)), y_true] = 1
        keep = Y.sum(0) > 0
        try:
            out["roc_auc_macro_ovr"] = float(
                roc_auc_score(Y[:, keep], y_prob[:, keep], average="macro"))
            out["roc_auc_weighted_ovr"] = float(
                roc_auc_score(Y[:, keep], y_prob[:, keep], average="weighted"))
            out["avg_precision_macro"] = float(
                average_precision_score(Y[:, keep], y_prob[:, keep], average="macro"))
        except Exception as e:
            out["roc_auc_macro_ovr"] = float("nan")
            out["auc_note"] = f"undefined: {e}"

    tn = ([str(c) for c in class_names] if class_names is not None
          else [str(i) for i in labels])
    out["_report_df"] = pd.DataFrame(classification_report(
        y_true, y_pred, labels=labels, target_names=tn,
        output_dict=True, zero_division=0)).T
    out["_confusion"] = confusion_matrix(y_true, y_pred, labels=labels)
    out["_labels"] = labels
    return out

def metrics_frame(list_of_metrics, sort_by="f1_macro"):
    """Tidy comparison table; drops the private '_' keys."""
    rows = [{k: v for k, v in m.items() if not k.startswith("_")}
            for m in list_of_metrics]
    df = pd.DataFrame(rows)
    cols = [c for c in ["model", "accuracy", "balanced_accuracy", "f1_macro",
                        "f1_weighted", "precision_macro", "recall_macro",
                        "precision_weighted", "recall_weighted",
                        "roc_auc_macro_ovr", "avg_precision_macro", "mcc",
                        "cohen_kappa", "params", "size_mb", "train_seconds",
                        "inference_ms_per_image", "n_test", "n_classes"]
            if c in df.columns]
    df = df[cols + [c for c in df.columns if c not in cols]]
    if sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=False)
    return df.reset_index(drop=True)

@torch.no_grad()
def measure_inference_ms(model, loader, device=None, n_batches=10):
    """Mean milliseconds per image at inference (brief asks for inference time)."""
    device = device or DEVICE
    model.eval().to(device)
    it = iter(loader)
    try:
        xb, _ = next(it)
        model(xb.to(device))
    except StopIteration:
        return float("nan")
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0, n = time.time(), 0
    for _ in range(n_batches):
        try:
            xb, _ = next(it)
        except StopIteration:
            break
        model(xb.to(device))
        n += xb.size(0)
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) * 1000.0 / max(1, n)

def _plt():
    import matplotlib
    if not on_kaggle():
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    return plt

SAVED_FIGURES: list[str] = []
_FIG_AUTO_SEQ = [0]

def _install_headless_hooks() -> None:
    import matplotlib
    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt
    from matplotlib.figure import Figure

    if getattr(plt, "_malebin_headless", False):
        return

    _orig_savefig = Figure.savefig

    def _savefig(self, fname, *a, **k):
        r = _orig_savefig(self, fname, *a, **k)
        try:
            self._malebin_saved = str(fname)
        except Exception:
            pass
        return r

    Figure.savefig = _savefig

    def _show(*a, **k):
        for num in plt.get_fignums():
            fig = plt.figure(num)
            path = getattr(fig, "_malebin_saved", None)
            if path is None:
                _FIG_AUTO_SEQ[0] += 1
                path = str(CFG.fig(f"auto_{_FIG_AUTO_SEQ[0]:03d}.png"))
                _orig_savefig(fig, path, dpi=130, bbox_inches="tight")
            SAVED_FIGURES.append(path)
            print(f"  [figure] {Path(path).name}")
            plt.close(fig)

    plt.show = _show
    plt._malebin_headless = True

    try:
        import plotly.graph_objects as go

        _orig_write_html = go.Figure.write_html

        def _write_html(self, *a, **k):
            r = _orig_write_html(self, *a, **k)
            if a:
                try:
                    self._malebin_saved = str(a[0])
                    SAVED_FIGURES.append(str(a[0]))
                except Exception:
                    pass
            return r

        def _pshow(self, *a, **k):
            path = getattr(self, "_malebin_saved", None)
            if path is None:
                _FIG_AUTO_SEQ[0] += 1
                path = str(CFG.fig(f"interactive_{_FIG_AUTO_SEQ[0]:03d}.html"))
                try:
                    _orig_write_html(self, path, include_plotlyjs="cdn")
                except Exception as e:
                    print(f"  [figure] plotly write_html failed: {e!r}")
                    return
                SAVED_FIGURES.append(path)
            print(f"  [figure] {Path(path).name}")

        go.Figure.write_html = _write_html
        go.Figure.show = _pshow
    except Exception:
        pass

    print("[malebin] headless figure mode ON -- plots go to "
          f"{CFG.path('figures')}, not into this notebook")

if os.environ.get("MALEBIN_NO_INLINE") in ("1", "true", "True"):
    _install_headless_hooks()

def plot_history(hist_df, title="Training curves", save=None):
    plt = _plt()
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(hist_df.epoch, hist_df.train_loss, marker="o")
    ax[0].set(title="Training loss", xlabel="epoch", ylabel="cross-entropy")
    ax[1].plot(hist_df.epoch, hist_df.train_acc, marker="o", label="train acc")
    ax[1].plot(hist_df.epoch, hist_df.val_acc, marker="s", label="val acc")
    ax[1].set(title="Accuracy", xlabel="epoch", ylabel="accuracy")
    ax[1].legend()
    ax[2].plot(hist_df.epoch, hist_df.val_macro_f1, marker="d", color="tab:green")
    b = hist_df.val_macro_f1.idxmax()
    ax[2].axvline(hist_df.epoch[b], ls="--", c="grey",
                  label=f"best ep {int(hist_df.epoch[b])} = {hist_df.val_macro_f1[b]:.4f}")
    ax[2].set(title="Validation macro-F1 (model-selection metric)",
              xlabel="epoch", ylabel="macro-F1")
    ax[2].legend()
    for a in ax:
        a.grid(alpha=.3)
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=130, bbox_inches="tight")
    return fig

def plot_confusion(cm, class_names, title="Confusion matrix", normalize=True,
                   save=None, figsize=None, annot_threshold=20):
    plt = _plt()
    cm = np.asarray(cm, dtype=float)
    if normalize:
        cm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1e-9, None)
    k = len(class_names)
    figsize = figsize or (max(7, k * 0.36), max(6, k * 0.32))
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(cm, cmap="viridis", vmin=0, vmax=1 if normalize else None)
    ax.set(xticks=range(k), yticks=range(k),
           xlabel="predicted family", ylabel="true family",
           title=title + (" (row-normalised)" if normalize else " (counts)"))
    ax.set_xticklabels(class_names, rotation=90, fontsize=7)
    ax.set_yticklabels(class_names, fontsize=7)
    if k <= annot_threshold:
        for i in range(k):
            for j in range(k):
                if cm[i, j] > 1e-3:
                    ax.text(j, i, f"{cm[i,j]:.2f}" if normalize else f"{int(cm[i,j])}",
                            ha="center", va="center", fontsize=6,
                            color="white" if cm[i, j] < 0.6 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046,
                 label="recall" if normalize else "count")
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=130, bbox_inches="tight")
    return fig

def plot_roc_ovr(y_true, y_prob, class_names, title="ROC (one-vs-rest)",
                 save=None, max_curves=40):
    from sklearn.metrics import roc_curve, auc
    plt = _plt()
    y_true = np.asarray(y_true)
    fig, ax = plt.subplots(figsize=(7.2, 6))
    aucs = []
    for i, cn in enumerate(class_names[:max_curves]):
        yi = (y_true == i).astype(int)
        if yi.sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(yi, y_prob[:, i])
        a = auc(fpr, tpr)
        aucs.append(a)
        ax.plot(fpr, tpr, lw=1.0, alpha=.75)
    grid = np.linspace(0, 1, 200)
    mean_tpr = np.zeros_like(grid)
    used = 0
    for i in range(len(class_names)):
        yi = (y_true == i).astype(int)
        if yi.sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(yi, y_prob[:, i])
        mean_tpr += np.interp(grid, fpr, tpr)
        used += 1
    mean_tpr /= max(1, used)
    ax.plot(grid, mean_tpr, "k-", lw=2.5,
            label=f"macro average (AUC = {np.mean(aucs):.4f})")
    ax.plot([0, 1], [0, 1], "r--", lw=1, label="chance")
    ax.set(xlabel="false positive rate", ylabel="true positive rate",
           title=f"{title}\nthin lines = individual families ({used} classes)")
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=.3)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=130, bbox_inches="tight")
    return fig, float(np.mean(aucs))

def plot_pr_ovr(y_true, y_prob, class_names, title="Precision-Recall (one-vs-rest)",
                save=None, max_curves=40):
    from sklearn.metrics import precision_recall_curve, average_precision_score
    plt = _plt()
    y_true = np.asarray(y_true)
    fig, ax = plt.subplots(figsize=(7.2, 6))
    aps = []
    for i, cn in enumerate(class_names[:max_curves]):
        yi = (y_true == i).astype(int)
        if yi.sum() == 0:
            continue
        pr, rc, _ = precision_recall_curve(yi, y_prob[:, i])
        aps.append(average_precision_score(yi, y_prob[:, i]))
        ax.plot(rc, pr, lw=1.0, alpha=.75)
    ax.axhline(1.0 / max(1, len(class_names)), ls="--", c="r", lw=1,
               label="chance (uniform prior)")
    ax.set(xlabel="recall", ylabel="precision",
           title=f"{title}\nmacro average precision = {np.mean(aps):.4f}")
    ax.legend(loc="lower left", fontsize=9)
    ax.grid(alpha=.3)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=130, bbox_inches="tight")
    return fig, float(np.mean(aps))

def mcnemar_test(y_true, pred_a, pred_b, name_a="A", name_b="B",
                 exact=None, correction=True, verbose=True):
    """
    McNemar's test on ONE shared held-out test set.

    This is the correct test for "does my model beat my baseline" when both
    models produced predictions for the same samples.  It can NOT be used
    against a paper's reported number (we never see their per-sample
    predictions) -- exactly the mistake the brief warns about in Sec. 6.4.

    Contingency table
        n11  both correct        n10  only A correct
        n01  only B correct      n00  both wrong
    H0: n10 and n01 come from the same distribution (models are equivalent).
    """
    from scipy import stats
    y_true = np.asarray(y_true)
    a = np.asarray(pred_a) == y_true
    b = np.asarray(pred_b) == y_true
    n11 = int(np.sum(a & b))
    n10 = int(np.sum(a & ~b))
    n01 = int(np.sum(~a & b))
    n00 = int(np.sum(~a & ~b))
    n_disc = n10 + n01
    if exact is None:
        exact = n_disc < 25
    if n_disc == 0:
        stat, p = 0.0, 1.0
        kind = "degenerate (no discordant pairs -> models made identical errors)"
    elif exact:
        stat = float(min(n10, n01))
        p = float(stats.binomtest(min(n10, n01), n_disc, 0.5,
                                  alternative="two-sided").pvalue)
        kind = "exact binomial"
    else:
        c = 1.0 if correction else 0.0
        stat = float((abs(n10 - n01) - c) ** 2 / n_disc)
        p = float(stats.chi2.sf(stat, df=1))
        kind = "chi-square" + (" with Yates correction" if correction else "")
    res = dict(test="McNemar", variant=kind, statistic=stat, p_value=p,
               n11=n11, n10=n10, n01=n01, n00=n00, n_discordant=n_disc,
               model_a=name_a, model_b=name_b,
               acc_a=float(a.mean()), acc_b=float(b.mean()),
               significant_at_0p05=bool(p < 0.05),
               better=(name_a if n10 > n01 else name_b if n01 > n10 else "tie"))
    if verbose:
        banner("McNemar test on the shared held-out test set", "-")
        print(f"  {name_a} accuracy = {a.mean():.4f}   {name_b} accuracy = {b.mean():.4f}")
        print(f"  contingency:  both correct {n11}   only {name_a} {n10}   "
              f"only {name_b} {n01}   both wrong {n00}")
        print(f"  variant     : {kind}")
        print(f"  statistic   = {stat:.4f}")
        print(f"  p-value     = {p:.6g}")
        print(f"  alpha = 0.05 -> "
              f"{'REJECT H0: the two models differ significantly' if p < 0.05 else 'FAIL TO REJECT H0: difference is not significant'}")
        print(f"  direction   : {res['better']} made more exclusive correct calls")
    return res

def wilcoxon_folds(scores_a, scores_b, name_a="A", name_b="B", verbose=True):
    """
    Wilcoxon signed-rank test over paired per-fold scores (5-fold CV).

    This is the right test when what we have is 5 paired fold scores, not
    per-sample predictions.  With n=5 the exact two-sided p-value cannot go
    below 0.0625, so we also report the one-sided p-value and Cohen's d, and we
    say so in the report rather than pretending n=5 gives strong evidence.
    """
    from scipy import stats
    a = np.asarray(scores_a, dtype=float)
    b = np.asarray(scores_b, dtype=float)
    assert a.shape == b.shape, "paired scores must have the same length"
    d = a - b
    res = dict(test="Wilcoxon signed-rank", n_folds=int(len(a)),
               model_a=name_a, model_b=name_b,
               mean_a=float(a.mean()), std_a=float(a.std(ddof=1)) if len(a) > 1 else 0.0,
               mean_b=float(b.mean()), std_b=float(b.std(ddof=1)) if len(b) > 1 else 0.0,
               mean_diff=float(d.mean()))
    if np.allclose(d, 0):
        res.update(statistic=0.0, p_value=1.0, p_value_one_sided=1.0,
                   note="all fold differences are zero")
    else:
        w = None
        for kw in ({"method": "exact"}, {"mode": "exact"}, {}):
            try:
                w = stats.wilcoxon(a, b, alternative="two-sided",
                                   zero_method="wilcox", **kw)
                break
            except TypeError:
                continue
        if w is None:
            w = stats.wilcoxon(a, b, alternative="two-sided")
        try:
            w1 = stats.wilcoxon(a, b, alternative="greater", zero_method="wilcox")
            p1 = float(w1.pvalue)
        except Exception:
            p1 = float("nan")
        res.update(statistic=float(w.statistic), p_value=float(w.pvalue),
                   p_value_one_sided=p1)
    if len(a) > 1 and not np.allclose(d, 0):
        t = stats.ttest_rel(a, b)
        res["paired_t_statistic"] = float(t.statistic)
        res["paired_t_p_value"] = float(t.pvalue)
        res["cohens_d_paired"] = float(d.mean() / (d.std(ddof=1) + 1e-12))
    res["significant_at_0p05"] = bool(res["p_value"] < 0.05)
    res["min_possible_two_sided_p"] = float(2 / 2 ** len(a)) if len(a) <= 10 else 0.0
    if verbose:
        banner(f"Wilcoxon signed-rank over {len(a)} folds: {name_a} vs {name_b}", "-")
        for i, (x, y) in enumerate(zip(a, b), 1):
            print(f"    fold {i}: {name_a} {x:.4f}   {name_b} {y:.4f}   diff {x-y:+.4f}")
        print(f"  {name_a}: {a.mean():.4f} +- {res['std_a']:.4f}")
        print(f"  {name_b}: {b.mean():.4f} +- {res['std_b']:.4f}")
        print(f"  W statistic = {res['statistic']:.4f}")
        print(f"  p (two-sided) = {res['p_value']:.6g}    "
              f"p (one-sided, A>B) = {res.get('p_value_one_sided', float('nan')):.6g}")
        if "paired_t_p_value" in res:
            print(f"  paired t-test: t = {res['paired_t_statistic']:.4f}, "
                  f"p = {res['paired_t_p_value']:.6g}, "
                  f"Cohen's d = {res['cohens_d_paired']:.3f}")
        print(f"  alpha = 0.05 -> "
              f"{'significant' if res['significant_at_0p05'] else 'NOT significant'}")
        print(f"  note: with n={len(a)} folds the smallest attainable two-sided "
              f"p is {res['min_possible_two_sided_p']:.4f}")
    return res

def friedman_nemenyi(score_matrix, model_names, verbose=True):
    """
    Friedman test + Nemenyi post-hoc for 3+ models over the same folds
    (Demsar, JMLR 2006).  `score_matrix` is (n_folds, n_models).

    Nemenyi critical difference:  CD = q_alpha * sqrt(k(k+1) / (6N))
    """
    from scipy import stats
    S = np.asarray(score_matrix, dtype=float)
    N, k = S.shape
    assert k == len(model_names) and k >= 3, "need >=3 models, one column each"
    chi2, p = stats.friedmanchisquare(*[S[:, j] for j in range(k)])
    ranks = np.apply_along_axis(lambda r: stats.rankdata(-r), 1, S)
    avg_rank = ranks.mean(axis=0)

    Q05 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949,
           8: 3.031, 9: 3.102, 10: 3.164}
    q = Q05.get(k, 3.164 + 0.03 * (k - 10))
    CD = q * math.sqrt(k * (k + 1) / (6.0 * N))

    pair = []
    for i in range(k):
        for j in range(i + 1, k):
            diff = abs(avg_rank[i] - avg_rank[j])
            pair.append(dict(model_a=model_names[i], model_b=model_names[j],
                             rank_diff=float(diff),
                             exceeds_CD=bool(diff > CD),
                             better=model_names[i] if avg_rank[i] < avg_rank[j]
                             else model_names[j]))
    res = dict(test="Friedman + Nemenyi", n_folds=int(N), n_models=int(k),
               chi2_statistic=float(chi2), p_value=float(p),
               significant_at_0p05=bool(p < 0.05),
               average_ranks={m: float(r) for m, r in zip(model_names, avg_rank)},
               critical_difference=float(CD), pairwise=pair)
    if verbose:
        banner(f"Friedman test over {N} folds, {k} models", "-")
        print(f"  chi2 = {chi2:.4f}   p = {p:.6g}   -> "
              f"{'at least one model differs' if p < 0.05 else 'no detectable difference'}")
        print("  average ranks (1 = best):")
        for m, r in sorted(res["average_ranks"].items(), key=lambda t: t[1]):
            print(f"    {r:5.2f}  {m}")
        print(f"  Nemenyi critical difference (alpha=0.05) = {CD:.3f}")
        for pr in pair:
            mark = "SIGNIFICANT" if pr["exceeds_CD"] else "not significant"
            print(f"    {pr['model_a']} vs {pr['model_b']}: "
                  f"|dRank| = {pr['rank_diff']:.2f}  -> {mark}")
        if N < 10:
            print(f"  note: Nemenyi is conservative with only N={N} folds; treat "
                  f"non-significant pairs as 'not resolved', not 'identical'.")
    return res

class GradCAM:
    """
    Minimal Grad-CAM (Selvaraju et al., ICCV 2017), no external dependency.

    cam = ReLU( sum_k  alpha_k * A_k ),   alpha_k = GAP( dY_c / dA_k )
    Grad-CAM++ weights are also available (alpha from second-order terms), which
    behaves better when several disjoint regions support the same class -- the
    common case for byte-plots.
    """

    def __init__(self, model, target_layer=None):
        self.model = model.eval()
        self.layer = target_layer if target_layer is not None else gradcam_target(model)
        self.acts = None
        self.grads = None
        self._h = [self.layer.register_forward_hook(self._fwd)]
        try:
            self._h.append(self.layer.register_full_backward_hook(self._bwd))
        except AttributeError:
            self._h.append(self.layer.register_backward_hook(self._bwd))

    def _fwd(self, m, i, o):
        self.acts = o.detach()

    def _bwd(self, m, gi, go):
        self.grads = go[0].detach()

    def remove(self):
        for h in self._h:
            h.remove()

    def __call__(self, x, class_idx=None, plus_plus=False):
        """x: (1,C,H,W) tensor. Returns (cam HxW in [0,1], class_idx, prob)."""
        self.model.zero_grad(set_to_none=True)
        was = torch.is_grad_enabled()
        torch.set_grad_enabled(True)
        try:
            logits = self.model(x)
            prob = torch.softmax(logits, 1)
            if class_idx is None:
                class_idx = int(logits.argmax(1).item())
            logits[0, class_idx].backward()
        finally:
            torch.set_grad_enabled(was)

        A, G = self.acts[0], self.grads[0]
        if plus_plus:
            g2, g3 = G.pow(2), G.pow(3)
            denom = 2 * g2 + g3 * A.sum(dim=(1, 2), keepdim=True)
            alpha = (g2 / denom.clamp(min=1e-9))
            w = (alpha * F.relu(G)).sum(dim=(1, 2))
        else:
            w = G.mean(dim=(1, 2))
        cam = F.relu((w[:, None, None] * A).sum(0))
        cam = cam - cam.min()
        cam = cam / cam.max().clamp(min=1e-9)
        cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode="bilinear",
                            align_corners=False)[0, 0]
        return cam.cpu().numpy(), class_idx, float(prob[0, class_idx].item())

def cam_to_byte_offsets(cam, img_width_px=256, top_frac=0.25):
    """
    Turn a Grad-CAM map into a statement about *file byte offsets*.

    Row r of a byte-plot with pixel width W holds bytes [r*W, (r+1)*W) of the
    original binary.  So the row-mean of the CAM is an importance profile over
    relative file position, and that IS interpretable (PE header at the top,
    code sections next, resources/padding at the end).

    Returns dict with the row profile, the contiguous top bands, and each band
    expressed as a percentage range of the file.
    """
    cam = np.asarray(cam, dtype=float)
    h = cam.shape[0]
    prof = cam.mean(axis=1)
    prof = (prof - prof.min()) / (np.ptp(prof) + 1e-9)
    thr = np.quantile(prof, 1.0 - top_frac)
    hot = prof >= thr
    bands, start = [], None
    for i, v in enumerate(hot):
        if v and start is None:
            start = i
        elif not v and start is not None:
            bands.append((start, i - 1))
            start = None
    if start is not None:
        bands.append((start, h - 1))
    out_bands = [dict(row_from=int(a), row_to=int(b),
                      file_pct_from=round(100.0 * a / h, 1),
                      file_pct_to=round(100.0 * (b + 1) / h, 1),
                      mean_importance=float(prof[a:b + 1].mean()))
                 for a, b in bands]
    out_bands.sort(key=lambda d: -d["mean_importance"])
    return dict(row_profile=prof, bands=out_bands,
                col_profile=(lambda c: (c - c.min()) / (np.ptp(c) + 1e-9))(cam.mean(axis=0)))

@torch.no_grad()
def occlusion_by_row_band(model, x, class_idx, n_bands=16, fill=0.0, device=None):
    """
    Causal check on the Grad-CAM story: blank out one horizontal band of the
    byte-plot at a time and record how far the target-class probability drops.
    A band = a contiguous slice of file offsets, so the result is directly
    comparable with `cam_to_byte_offsets`.

    Returns (drops array of length n_bands, baseline probability).
    """
    device = device or DEVICE
    model.eval().to(device)
    x = x.to(device)
    base = torch.softmax(model(x), 1)[0, class_idx].item()
    H = x.shape[-2]
    edges = np.linspace(0, H, n_bands + 1).astype(int)
    drops = np.zeros(n_bands)
    for i in range(n_bands):
        xo = x.clone()
        xo[..., edges[i]:edges[i + 1], :] = fill
        p = torch.softmax(model(xo), 1)[0, class_idx].item()
        drops[i] = base - p
    return drops, base

def lime_explain(model, img_uint8, class_names, in_channels=1, out_size=None,
                 top_labels=3, num_samples=400, num_features=8,
                 segments="grid", device=None, seed=None):
    """
    LIME image explanation (Ribeiro et al., KDD 2016).

    Segmentation choice, and why it matters here
    -------------------------------------------
    LIME's default `quickshift` segmentation looks for *colour-coherent
    regions*, which is a natural-image assumption.  A byte-plot has no such
    regions, so quickshift produces essentially arbitrary blobs and the
    explanation is noise dressed up as insight.  We therefore default to a
    regular ROW-BAND grid: every superpixel is a contiguous slice of file
    offsets, which is the only partition of a byte-plot that has a meaning.
    Pass segments='quickshift' to reproduce the naive version for comparison.
    """
    from lime import lime_image
    device = device or DEVICE
    out_size = int(out_size or CFG.img_size)
    seed = CFG.seed if seed is None else seed
    model.eval().to(device)

    a = np.asarray(img_uint8, dtype=np.uint8)
    rgb = np.stack([a] * 3, axis=-1)

    def batch_predict(images):
        xs = []
        for im in images:
            g = im[..., 0].astype(np.float32) / 255.0
            t = torch.from_numpy(g)[None, None]
            if t.shape[-1] != out_size:
                t = F.interpolate(t, size=(out_size, out_size), mode="bilinear",
                                  align_corners=False)
            t = (t - t.mean()) / (t.std() + 1e-5)
            if in_channels == 3:
                t = t.repeat(1, 3, 1, 1)
            xs.append(t[0])
        xb = torch.stack(xs).to(device)
        with torch.no_grad():
            return torch.softmax(model(xb), 1).cpu().numpy()

    if segments == "grid":
        n_rows, n_cols = 16, 4
        H, W = a.shape
        seg = np.zeros((H, W), dtype=int)
        re_ = np.linspace(0, H, n_rows + 1).astype(int)
        ce_ = np.linspace(0, W, n_cols + 1).astype(int)
        k = 0
        for i in range(n_rows):
            for j in range(n_cols):
                seg[re_[i]:re_[i + 1], ce_[j]:ce_[j + 1]] = k
                k += 1
        seg_fn = (lambda image: seg)
    else:
        seg_fn = None

    expl = lime_image.LimeImageExplainer(random_state=seed)
    kw = dict(top_labels=top_labels, hide_color=0, num_samples=int(num_samples),
              batch_size=32)
    if seg_fn is not None:
        kw["segmentation_fn"] = seg_fn
    ex = expl.explain_instance(rgb, batch_predict, **kw)
    return ex

RELATED_WORK = [
    dict(
        key="PAFE2024",
        title="PAFE: A lightweight visualization-based fast malware classification method",
        authors="S. Li, J. Wang, S. Wang, Y. Song",
        year=2024, venue="Heliyon 10, e35965", doi="10.1016/j.heliyon.2024.e35965",
        dataset="Malimg (25 families, 9,435 images)",
        application="Windows PE malware family classification from grayscale byte-plots",
        method="CNN with FFSE blocks = multi-scale feature fusion + Squeeze-and-Excitation "
               "channel attention; pixel-padding resize instead of interpolation; 256x256 input",
        attention_type="Channel (Squeeze-and-Excitation) inside a multi-scale fusion block",
        metrics="Accuracy 99.25%, Precision 99.29%, Recall 99.25%, F1 99.27%, "
                "inference 10.04 ms, 721,913 params",
        headline_metric="F1", headline_value=99.27,
        strengths="Best published Malimg accuracy/latency trade-off; tiny (0.72M params); "
                  "pixel-padding avoids the texture distortion that bilinear resizing causes; "
                  "reports timing and parameter count, not just accuracy",
        limitations="Random (not source-grouped) split, so polymorphic near-duplicates can "
                    "straddle train/test; no macro-F1 or per-class recall on the rare families; "
                    "authors themselves note generalisation to new variants is unverified; "
                    "no explainability",
        gap="Leakage-controlled evaluation and per-class (macro) reporting are missing; "
            "attention is channel-only, so byte-offset position is discarded",
        relation="Our Pillar-A target. We reproduce the same Malimg 25-family task as a "
                 "subset of MaleBin and compare F1, but under a duplicate-grouped split",
        comparable=True,
    ),
    dict(
        key="DRIN2024",
        title="Attention-Based Malware Detection Model by Visualizing Latent Features "
              "Through Dynamic Residual Kernel Network",
        authors="M. Basak, D.-W. Kim, M.-M. Han, G.-Y. Shin",
        year=2024, venue="Sensors 24(24), 7953", doi="10.3390/s24247953",
        dataset="Custom (25 families, 49,374), Malimg, MaleVis",
        application="Malware family classification from visualised binaries",
        method="Dynamic Residual Involution Network (DRIN): involution kernels that are "
               "spatially specific and channel-agnostic, i.e. attention baked into the kernel",
        attention_type="Involution (spatially specific, channel-agnostic) + residual",
        metrics="Malimg: Acc 99.3%, P 0.992, R 0.989, F1 0.9905 | "
                "MaleVis: Acc 98.9%, F1 0.9892 | Custom: Acc 99.5%, F1 0.9948",
        headline_metric="F1", headline_value=99.05,
        strengths="Attention is intrinsic to the kernel rather than bolted on; validated on "
                  "three datasets; heat-map visualisation attempted",
        limitations="Authors state it still struggles on under-represented classes; heavier "
                    "than lightweight CNNs; sensitive to preprocessing noise; scalability to "
                    "unseen families untested; interpretability admitted to be limited",
        gap="The under-represented-class weakness is exactly what macro-F1 exposes and what "
            "class-weighted training plus a balanced dataset can address",
        relation="Second comparable Malimg baseline; its admitted rare-class weakness "
                 "motivates our macro-F1-driven model selection and class-weighted loss",
        comparable=True,
    ),
    dict(
        key="SEAGM2023",
        title="Transfer Learning for Image-Based Malware Detection for IoT (SE-AGM)",
        authors="P. Panda, C. U. Om Kumar, S. Marappan, M. Suresh, S. Manimurugan, "
                "D. Veesani Nandi",
        year=2023, venue="Sensors 23(6), 3253", doi="10.3390/s23063253",
        dataset="Malimg (25 families)",
        application="IoT malware detection from byte-plot images",
        method="Stacked ensemble of autoencoder + GRU + MLP over 25 CNN-extracted features; "
               "each stage's output feeds the next; data augmentation studied",
        attention_type="None (stacked ensemble, not attention)",
        metrics="Average accuracy 99.43% on Malimg",
        headline_metric="Accuracy", headline_value=99.43,
        strengths="Highest reported Malimg accuracy in our set; very cheap at inference "
                  "because it classifies only 25 encoded features; ablates augmentation",
        limitations="Reports accuracy only -- no macro-F1, no per-class recall, no confusion "
                    "matrix on the rare families, so the number cannot be checked against "
                    "the imbalance rule; feature extractor trained on the same data it later "
                    "encodes; random split",
        gap="An accuracy-only headline on an imbalanced 25-class set is exactly what Sec. 6.2 "
            "warns about; the result is not verifiable per class",
        relation="Highest accuracy number we must acknowledge, but it is accuracy-only, so we "
                 "compare our accuracy to it and explain why macro-F1 is the fairer basis",
        comparable=True,
    ),
    dict(
        key="Hybrid2025",
        title="An Explainable Hybrid CNN-Transformer Architecture for Visual Malware "
              "Classification",
        authors="M. Alshomrani, A. Albeshri, A. A. Alsulami, B. Alturki",
        year=2025, venue="Sensors 25(15), 4581", doi="10.3390/s25154581",
        dataset="Malimg + MaleVis + VirusMNIST combined (61 classes); also Maldeb, "
                "Dumpware-10",
        application="Visual malware classification across merged sources",
        method="ConvNeXt-Tiny (local features) fused with Swin Transformer (global context); "
               "Grad-CAM for interpretability; real-time deployment demo",
        attention_type="Self-attention (shifted-window Swin) + convolutional local features",
        metrics="Combined 61-class validation accuracy 94.04% (ConvNeXt-Tiny alone 92.45%, "
                "Swin alone 90.44%); Maldeb 98%; Dumpware-10 97%",
        headline_metric="Accuracy", headline_value=94.04,
        strengths="The only paper in our set that evaluates a *merged multi-source* label "
                  "space, which is what MaleBin is; uses Grad-CAM and discusses it; shows "
                  "the hybrid beats either half",
        limitations="Accuracy on a validation split rather than an untouched test set; no "
                    "macro-F1 on 61 imbalanced classes; Grad-CAM interpreted without "
                    "acknowledging that byte-plots have no semantic regions; heavy backbones",
        gap="Merged-source label spaces drop ~5 points versus single-source Malimg, and nobody "
            "reports macro-F1 there; also no duplicate control across merged sources",
        relation="Closest analogue to our 39-class MaleBin setting (merged sources, more "
                 "classes). This is the paper our full-MaleBin number is compared against",
        comparable=True,
    ),
    dict(
        key="MalVis2025",
        title="MalVis: A Large-Scale Image-Based Framework and Dataset for Advancing "
              "Android Malware Classification",
        authors="S. J. Makkawy, M. J. De Lucia, K. E. Barner",
        year=2025, venue="arXiv:2505.12106", doi="10.48550/arXiv.2505.12106",
        dataset="MalVis (>1.3M images, 9 malware classes + benign)",
        application="Android malware classification from bytecode visualisations",
        method="Entropy + N-gram enhanced visualisation; MobileNetV2 / DenseNet201 / "
               "ResNet50 / InceptionV3 with eight ensemble strategies; undersampling",
        attention_type="None",
        metrics="Accuracy 95.19%, macro-F1 90.81%, Precision 92.58%, Recall 89.10%, "
                "MCC 87.58%, ROC-AUC 98.06%",
        headline_metric="macro-F1", headline_value=90.81,
        strengths="Reports macro-F1, MCC and ROC-AUC -- the honest metric set for imbalanced "
                  "data; huge scale; explicit imbalance handling",
        limitations="Android bytecode, not Windows PE byte-plots, so not directly comparable; "
                    "undersampling discards data; no attention module",
        gap="Shows how far macro-F1 sits below accuracy on imbalanced visual malware data "
            "(95.19 vs 90.81) -- a gap the PE-image papers never report",
        relation="Context, not a comparison target. It is our evidence that macro-F1 is the "
                 "right headline metric and that accuracy overstates performance",
        comparable=False,
    ),
    dict(
        key="Byteplot2023",
        title="Comparative Analysis of Imbalanced Malware Byteplot Image Classification "
              "using Transfer Learning",
        authors="Jayasudha M, A. Shaik, G. Pendharkar, S. Kumar, Muhesh Kumar B, "
                "S. Balaji",
        year=2023, venue="PEIS 2023, Lecture Notes in Electrical Engineering; "
                         "arXiv:2310.02742",
        doi="10.48550/arXiv.2310.02742",
        dataset="Malimg, a blended dataset, and MaleVis (three imbalance levels)",
        application="Byte-plot malware classification under class imbalance",
        method="Six multi-class transfer-learning models; ResNet50, EfficientNetB0 and "
               "DenseNet169 were the strongest",
        attention_type="None (pure transfer learning)",
        metrics="Max precision 97% (imbalanced), 95% (intermediate), 95% (balanced); "
                "more imbalance -> faster convergence but higher variance across models",
        headline_metric="Precision", headline_value=97.0,
        strengths="Directly studies the imbalance axis; uses a blended (multi-source) dataset "
                  "like MaleBin; documents the convergence/variance trade-off",
        limitations="Precision-only headline; no macro-F1 or per-class recall; no attention "
                    "or custom architecture; no duplicate control across the blend",
        gap="Blended multi-source byte-plot data is under-studied and reported with the wrong "
            "metric",
        relation="Justifies our baseline pool (ResNet50, DenseNet121, EfficientNet-B0) and our "
                 "decision to report variance across folds, not a single number",
        comparable=False,
    ),
    dict(
        key="IMCMK2024",
        title="IMCMK-CNN: A lightweight convolutional neural network with Multi-scale "
              "Kernels for Image-based Malware Classification",
        authors="D. Zhang, Y. Song, Q. Xiang, Y. Wang",
        year=2024, venue="Alexandria Engineering Journal 111, 203-220",
        doi="10.1016/j.aej.2024.10.055",
        dataset="Malimg and other image-based malware sets",
        application="Malware variant classification from byte-plot images",
        method="Multi-scale Kernel (MK) block mixing large and small kernels plus an improved "
               "Squeeze-and-Excitation block; fusion strategy keeps the parameter cost of "
               "small kernels",
        attention_type="Improved Squeeze-and-Excitation channel attention",
        metrics="[FILL FROM PDF -- read the results table of the published version and "
                "replace this string before you submit Task 1]",
        headline_metric="Accuracy", headline_value=float("nan"),
        strengths="Directly motivates multi-scale kernels for byte texture; explicitly targets "
                  "the parameter cost of large kernels",
        limitations="Channel attention only, so byte-offset position is not modelled; "
                    "single-source evaluation",
        gap="Multi-scale + channel attention is established; direction-aware (positional) "
            "attention for byte-plots is not",
        relation="The architectural ancestor of our multi-scale dilated block. We keep its "
                 "multi-scale idea and add the positional attention it lacks",
        comparable=True,
    ),
]

def related_work_frame(only_comparable=False):
    cols = ["key", "title", "authors", "year", "venue", "doi", "dataset",
            "application", "method", "attention_type", "metrics",
            "headline_metric", "headline_value", "strengths", "limitations",
            "gap", "relation", "comparable"]
    rows = [r for r in RELATED_WORK if (r["comparable"] or not only_comparable)]
    return pd.DataFrame(rows)[cols]

def best_comparable_target(scope=None):
    """
    The single number Pillar A is judged against, chosen honestly per scope.

    scope='malimg25'  -> PAFE 2024, F1 99.27 on the very same 25 families.
    scope='malebin39' -> Alshomrani et al. 2025, 94.04% accuracy on a merged
                         61-class multi-source visual-malware label space, the
                         closest published analogue to MaleBin's merged 39.
    Anything else would be an unfair comparison under Sec. 6.4.
    """
    scope = scope or CFG.eval_scope
    if scope == "malimg25":
        r = next(x for x in RELATED_WORK if x["key"] == "PAFE2024")
        return dict(paper=r["key"], citation=f"{r['authors']} ({r['year']}), {r['venue']}",
                    metric="F1 (weighted) on Malimg 25 families", value=99.27,
                    caveat="Their split is random; ours is duplicate-grouped, which is "
                           "strictly harder. Same dataset, same 25 classes, same input type.")
    r = next(x for x in RELATED_WORK if x["key"] == "Hybrid2025")
    return dict(paper=r["key"], citation=f"{r['authors']} ({r['year']}), {r['venue']}",
                metric="Accuracy on a merged 61-class visual-malware set", value=94.04,
                caveat="Different merged corpus (Malimg+MaleVis+VirusMNIST, 61 classes) vs "
                       "MaleBin (39 classes). Not the same dataset, so this is an "
                       "indicative comparison and we label it as such. The strictly fair "
                       "comparison for this project is the malimg25 scope.")

In [ ]:
import os, sys, json, time, math, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": .3, "font.size": 10})

import malebin_common as M
from malebin_common import CFG

CFG.group = "Group00"
CFG.dataset_slug = "MaleBin"
CFG.__post_init__()
M.set_seed(CFG.seed)

print("python      :", sys.version.split()[0])
import torch
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))
print("numpy       :", np.__version__, "| pandas:", pd.__version__)
print("on kaggle   :", M.on_kaggle())
print("output dir  :", CFG.out_dir)
print("FAST mode   :", CFG.fast, "(set MALEBIN_FAST=1 to enable)")
print("img_size    :", CFG.img_size, "| cache", CFG.cache_size,
      "| epochs", CFG.epochs, "| folds", CFG.n_folds)
print("eval scope  :", CFG.eval_scope)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=False)

from scipy import stats as sstats
from PIL import Image
print("plotly ready")

---
## A · Summary table

We index every file, then read each image's real header (not the folder name) so
that resolution, mode and corruption are measured rather than assumed.

In [ ]:
ROOT = M.find_dataset_root()
df = M.scan_index(ROOT)

rows = []
bad = []
for p in df.path:
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            rows.append((im.size[0], im.size[1], im.mode, im.format))
    except Exception as e:
        bad.append((p, f"{type(e).__name__}: {e}"))
        rows.append((np.nan, np.nan, "CORRUPT", "?"))

df[["width", "height", "mode", "format"]] = pd.DataFrame(rows, index=df.index)
df["megapixels"] = df.width * df.height / 1e6
print(f"unreadable / corrupt files: {len(bad)}")
for p, e in bad[:10]:
    print("   ", p, "->", e)
df.head()

In [ ]:
vc = df.family.value_counts()
summary = pd.DataFrame([
    ("Dataset name",        "MaleBin: Malware Binary Greyscale Images"),
    ("Source",              "Kaggle · tashiee/malebin-malware-binary-greyscale-images (CC BY 4.0)"),
    ("Compiled from",       "Malimg (Nataraj et al. 2011) + subset of kaggle/walt30/malware-images (MalwareBazaar)"),
    ("Application area",    "Static malware family triage from byte-plot images (cyber security)"),
    ("Problem type",        f"Single-label multi-class image classification ({df.family.nunique()} classes)"),
    ("Total samples",       f"{len(df):,}"),
    ("Classes (families)",  f"{df.family.nunique()}"),
    ("  of which Malimg",   f"{df.loc[df.is_malimg,'family'].nunique()} families / {df.is_malimg.sum():,} images"),
    ("  of which other",    f"{df.loc[~df.is_malimg,'family'].nunique()} families / {(~df.is_malimg).sum():,} images"),
    ("File format",         ", ".join(f"{k} ({v:,})" for k, v in df.format.value_counts().items())),
    ("Colour mode",         ", ".join(f"{k} ({v:,})" for k, v in df["mode"].value_counts().items())),
    ("Resolution",          ", ".join(f"{int(w)}x{int(h)} ({n:,})" for (w, h), n
                                      in df.groupby(["width","height"]).size().items())),
    ("Channels",            "1 (8-bit grayscale) — a pixel IS a byte value 0-255"),
    ("Samples / class",     f"min {vc.min()}  median {int(vc.median())}  max {vc.max()}"),
    ("Imbalance ratio",     f"{vc.max()/max(vc.min(),1):.2f} : 1  (majority : minority)"),
    ("Missing values",      "N/A for images — measured instead as unreadable files"),
    ("Corrupt / unreadable",f"{len(bad)}"),
    ("Total size on disk",  f"{df.n_bytes.sum()/1e6:,.0f} MB"),
    ("Subject / source id", "NOT provided by the uploader — we derive one "
                            "(see section F: duplicate groups)"),
], columns=["Property", "Value"])

pd.set_option("display.max_colwidth", 130)
display(summary)
summary.to_csv(CFG.art("task1_A_summary_table.csv"), index=False)

**Reading.** Uniform 256×256 8-bit grayscale, one channel, so no resizing or
colour handling is needed before modelling — a real convenience. The two facts
that shape every later decision are (i) the ~*N* : 1 class imbalance printed
above, which forces **macro-F1** as the headline metric (brief §6.2), and
(ii) the absence of any subject/source identifier, which means we must **derive
our own grouping key** before we are allowed to split the data (section F).

---
## B · Class balance and what it does to the metrics

In [ ]:
vc = df.family.value_counts().sort_values(ascending=False)
share = 100 * vc / vc.sum()

fig, ax = plt.subplots(2, 1, figsize=(13, 9),
                       gridspec_kw=dict(height_ratios=[2, 1]))
IS_MALIMG = df.drop_duplicates("family").set_index("family").is_malimg.to_dict()
colors = ["tab:blue" if IS_MALIMG.get(f, False) else "tab:orange" for f in vc.index]
ax[0].bar(range(len(vc)), vc.values, color=colors)
ax[0].axhline(vc.mean(), ls="--", c="k", lw=1, label=f"mean = {vc.mean():.0f}")
ax[0].axhline(vc.median(), ls=":", c="r", lw=1, label=f"median = {vc.median():.0f}")
ax[0].set_xticks(range(len(vc)))
ax[0].set_xticklabels(vc.index, rotation=90, fontsize=8)
ax[0].set(ylabel="number of images",
          title=f"B · Images per malware family (n={len(df):,}, "
                f"{len(vc)} classes)\nblue = one of the 25 original Malimg "
                f"families, orange = added from the MalwareBazaar-derived source")
ax[0].legend()

ax[1].plot(range(len(vc)), np.cumsum(share.values), marker="o", ms=4)
ax[1].axhline(80, ls="--", c="r", lw=1, label="80% of the data")
ax[1].set(xlabel="families, most frequent first", ylabel="cumulative % of images",
          title="Cumulative share — how concentrated is the dataset?")
ax[1].legend()
fig.tight_layout()
fig.savefig(CFG.fig("task1_B_class_balance.png"), dpi=130, bbox_inches="tight")
plt.show()

ratio = vc.max() / max(vc.min(), 1)
gini = 1 - ((share / 100) ** 2).sum() * len(vc) / (len(vc) - 1) + 1 / (len(vc) - 1)
print(f"majority class : {vc.index[0]}  ({vc.iloc[0]:,} images, {share.iloc[0]:.1f}%)")
print(f"minority class : {vc.index[-1]} ({vc.iloc[-1]:,} images, {share.iloc[-1]:.1f}%)")
print(f"imbalance ratio: {ratio:.2f} : 1")
print(f"a majority-class-only classifier would score "
      f"accuracy = {share.iloc[0]/100:.4f} but macro-F1 = "
      f"{(2*share.iloc[0]/100)/(1+share.iloc[0]/100)/len(vc):.4f}")
print(f"\nfamilies holding 80% of the data: "
      f"{int((np.cumsum(share.values) < 80).sum())+1} of {len(vc)}")
display(pd.DataFrame({"images": vc, "share_%": share.round(2),
                      "is_malimg": [IS_MALIMG.get(f, False) for f in vc.index]}))

**Reading.** The uploader deliberately trimmed families to balance them, so
MaleBin is far flatter than raw Malimg (which is ~2949 : 42). It is still not
uniform, and that has two consequences we carry through the whole project:

1. **Accuracy is not a safe headline.** A classifier that only ever predicted
   the majority family would score the accuracy printed above while getting
   38 of 39 families completely wrong. Per §6.2 we therefore report
   **macro-F1 and per-class recall** as the primary numbers, and we select
   models on *validation macro-F1*, never on accuracy.
2. **The loss must be re-weighted.** We use inverse-frequency class weights so
   rare families are not traded away for majority-class gains, and we verify in
   the Task-3 ablation that this actually helps rather than assuming it.

---
## C · Labelled sample grid — which families look alike?

In [ ]:
rng = np.random.default_rng(CFG.seed)
fams = sorted(df.family.unique())
n = len(fams)
ncol = 8
nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(2.05 * ncol, 2.35 * nrow))
for ax, fam in zip(axes.ravel(), fams):
    p = df.loc[df.family == fam, "path"].sample(1, random_state=CFG.seed).iloc[0]
    with Image.open(p) as im:
        ax.imshow(np.asarray(im.convert("L")), cmap="gray", vmin=0, vmax=255)
    tag = "M" if df.loc[df.family == fam, "is_malimg"].iloc[0] else "B"
    ax.set_title(f"{fam}\n[{tag}] n={int((df.family==fam).sum())}", fontsize=7.5)
    ax.axis("off")
for ax in axes.ravel()[n:]:
    ax.axis("off")
fig.suptitle("C · One random byte-plot per family  "
             "([M] = Malimg source, [B] = MalwareBazaar-derived source)\n"
             "row index = byte offset in the file; brightness = byte value",
             fontsize=12)
fig.tight_layout()
fig.savefig(CFG.fig("task1_C_sample_grid.png"), dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
pick = list(vc.index[:3]) + list(vc.index[-3:])
fig, axes = plt.subplots(len(pick), 6, figsize=(13, 2.2 * len(pick)))
for r, fam in enumerate(pick):
    ps = df.loc[df.family == fam, "path"]
    ps = ps.sample(min(6, len(ps)), random_state=CFG.seed).tolist()
    for c in range(6):
        ax = axes[r, c]
        if c < len(ps):
            with Image.open(ps[c]) as im:
                ax.imshow(np.asarray(im.convert("L")), cmap="gray", vmin=0, vmax=255)
        ax.axis("off")
        if c == 0:
            ax.set_title(f"{fam}  (n={int((df.family==fam).sum())})",
                         fontsize=9, loc="left")
fig.suptitle("C · Within-family variability — 3 largest families (top) and "
             "3 smallest (bottom)", fontsize=12)
fig.tight_layout()
fig.savefig(CFG.fig("task1_C_within_family.png"), dpi=120, bbox_inches="tight")
plt.show()

**Reading.** Two things stand out and both drive later design choices.

* **Byte-plots have banded, not object-like, structure.** Every image is a stack
  of horizontal bands of differing texture: dense speckle where executable code
  sits, strictly periodic patterns where a table or padding repeats, uniform
  black where the file is zero-padded, and high-entropy noise where a section is
  packed or encrypted. The *vertical position* of a band is meaningful — row
  *r* of a 256-px-wide plot holds bytes `[256r, 256r+256)` — while horizontal
  position is an artefact of the fixed row width. This is the single most
  important observation in the whole EDA: it is why we choose **coordinate
  attention** (a per-row gate) in Task 2, and why we forbid flips and rotations
  in augmentation.
* **Families in the same lineage look nearly identical** (see e.g. the
  `Allaple.A` / `Allaple.L` and `Swizzor.gen!E` / `Swizzor.gen!I` pairs, and the
  repeated rows within a single family). That near-identity is a *leakage
  hazard*, quantified in section F.

---
## D · Size / resolution audit

In [ ]:
print(df.groupby(["width", "height", "mode", "format"]).size().rename("count").to_frame())
print("\nfile size on disk (KB): compressed PNG size varies with byte entropy,")
print("so it is itself a weak signal about how packed a sample is.")
display(df.groupby("family").n_bytes.describe()[["count","mean","std","min","50%","max"]]
          .div(1024).round(1).rename(columns=lambda c: c + "_KB" if c != "count" else c))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].hist(df.n_bytes / 1024, bins=60, color="tab:purple")
ax[0].set(xlabel="PNG file size (KB)", ylabel="images",
          title="D · Compressed file-size distribution")
order = df.groupby("family").n_bytes.median().sort_values().index
ax[1].boxplot([df.loc[df.family == f, "n_bytes"] / 1024 for f in order],
              showfliers=False)
ax[1].set_xticks(range(1, len(order) + 1))
ax[1].set_xticklabels(order, rotation=90, fontsize=7)
ax[1].set(ylabel="PNG file size (KB)",
          title="D · File size by family (a proxy for byte entropy)")
fig.tight_layout()
fig.savefig(CFG.fig("task1_D_sizes.png"), dpi=130, bbox_inches="tight")
plt.show()

**Reading.** Every image is already 256×256×1, so there is no resolution
heterogeneity to fix — unusual and helpful. Note, though, that the uploader
reached that uniformity by **resizing** the original byte-plots, and warns this
distorts texture; a 256×256 image is 65,536 bytes, so any binary larger than
64 KB has been downsampled and any smaller one upsampled. That is a real
information loss we cannot undo, and we state it as a limitation rather than
pretending the pixels are raw bytes.

Compressed PNG size does still vary a lot between families, which is expected:
PNG compresses low-entropy regions well, so a packed/encrypted sample stays
large. It is a legitimate weak feature but we deliberately **do not** feed it to
the model — file size is metadata, not content, and using it would be the
image-domain equivalent of the ID-column leak §6.2 warns about.

---
## E · Pixel and texture statistics

For the statistical work (§B of the tabular EDA spec, adapted to images) we
derive a compact numeric feature vector per image. These features are used **for
EDA and visualisation only** — the CNN sees raw pixels.

In [ ]:
EDA_SIDE = 64 if CFG.fast else 128
imgs_eda = M.load_images(df, side=EDA_SIDE)
print("cache:", imgs_eda.shape, imgs_eda.dtype)

In [ ]:
def _safe_corr(u, v) -> float:
    """
    Pearson r that degrades to 0.0 instead of NaN.

    rows.std() > 0 is NOT enough to make corrcoef safe: a byte-plot whose rows
    are constant except for the very first or very last one leaves one of the
    two shifted halves with zero variance, and corrcoef then returns NaN.  Four
    real MaleBin images do exactly that, and a single NaN here propagates into
    StandardScaler -> PCA / t-SNE / UMAP / kNN and takes section G down with it.
    """
    if u.std() == 0 or v.std() == 0:
        return 0.0
    r = float(np.corrcoef(u, v)[0, 1])
    return 0.0 if not np.isfinite(r) else r

def image_features(a: np.ndarray) -> dict:
    """Interpretable per-image descriptors (byte-statistics + texture)."""
    x = a.astype(np.float32)
    flat = x.ravel()
    hist = np.bincount(a.ravel(), minlength=256).astype(np.float64)
    p = hist / hist.sum()
    nz = p[p > 0]
    gy, gx = np.gradient(x)
    rows = x.mean(axis=1)
    return dict(
        mean=flat.mean(), median=float(np.median(flat)), std=flat.std(),
        vmin=flat.min(), vmax=flat.max(),
        q25=float(np.percentile(flat, 25)), q50=float(np.percentile(flat, 50)),
        q75=float(np.percentile(flat, 75)),
        iqr=float(np.percentile(flat, 75) - np.percentile(flat, 25)),
        skewness=float(sstats.skew(flat)), kurtosis=float(sstats.kurtosis(flat)),
        entropy=float(-(nz * np.log2(nz)).sum()),
        zero_frac=float((a == 0).mean()),
        high_frac=float((a > 240).mean()),
        ascii_frac=float(((a >= 32) & (a <= 126)).mean()),
        grad_mag=float(np.hypot(gx, gy).mean()),
        row_var=float(rows.var()),
        row_autocorr=_safe_corr(rows[:-1], rows[1:]),
        col_var=float(x.mean(axis=0).var()),
        top_band_mean=float(x[: x.shape[0] // 8].mean()),
        bottom_band_mean=float(x[-x.shape[0] // 8:].mean()),
    )

t0 = time.time()
feat = pd.DataFrame([image_features(imgs_eda[i]) for i in range(len(df))])

_bad = (~np.isfinite(feat.select_dtypes("number"))).sum()
_bad = _bad[_bad > 0]
if len(_bad):
    print("[!] non-finite feature values repaired with the column median:")
    for _c, _n in _bad.items():
        print(f"      {_c}: {_n}")
    _num = feat.select_dtypes("number").columns
    feat[_num] = (feat[_num].replace([np.inf, -np.inf], np.nan)
                            .fillna(feat[_num].median(numeric_only=True)))
else:
    print("all per-image features are finite")
feat.insert(0, "family", df.family.values)
feat.insert(1, "label", df.label.values)
feat.insert(2, "is_malimg", df.is_malimg.values)
print(f"{feat.shape[1]-3} features x {len(feat):,} images in {time.time()-t0:.1f}s")
FEATCOLS = [c for c in feat.columns if c not in ("family", "label", "is_malimg")]
feat.to_csv(CFG.art("task1_E_image_features.csv"), index=False)

desc = feat[FEATCOLS].describe().T
desc["skewness"] = feat[FEATCOLS].skew()
desc["kurtosis"] = feat[FEATCOLS].kurtosis()
desc["n_outliers_1.5IQR"] = [
    int(((feat[c] < feat[c].quantile(.25) - 1.5 * (feat[c].quantile(.75) - feat[c].quantile(.25))) |
         (feat[c] > feat[c].quantile(.75) + 1.5 * (feat[c].quantile(.75) - feat[c].quantile(.25)))).sum())
    for c in FEATCOLS]
display(desc.round(3))
desc.round(4).to_csv(CFG.art("task1_E_feature_stats.csv"))

In [ ]:
show = ["mean", "std", "entropy", "zero_frac", "ascii_frac", "grad_mag",
        "row_var", "row_autocorr", "skewness", "kurtosis", "top_band_mean",
        "bottom_band_mean"]
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, c in zip(axes.ravel(), show):
    ax.hist(feat[c], bins=50, color="tab:blue")
    ax.set(title=f"{c}\nskew={feat[c].skew():.2f}  kurt={feat[c].kurtosis():.2f}",
           xlabel=c, ylabel="images")
fig.suptitle("E · Distributions of the derived byte/texture features", fontsize=13)
fig.tight_layout()
fig.savefig(CFG.fig("task1_E_histograms.png"), dpi=125, bbox_inches="tight")
plt.show()

In [ ]:
key = ["entropy", "std", "zero_frac", "row_autocorr"]
order = feat.groupby("family").entropy.median().sort_values().index.tolist()
fig, axes = plt.subplots(len(key), 1, figsize=(14, 3.3 * len(key)), sharex=True)
for ax, c in zip(axes, key):
    data = [feat.loc[feat.family == f, c].values for f in order]
    parts = ax.violinplot(data, showmedians=True, widths=.9)
    for pc in parts["bodies"]:
        pc.set_alpha(.55)
    ax.boxplot(data, widths=.18, showfliers=False,
               medianprops=dict(color="k", lw=1.2))
    grand = feat[c].mean()
    ssb = sum(len(d) * (d.mean() - grand) ** 2 for d in data if len(d))
    sst = ((feat[c] - grand) ** 2).sum()
    ax.set(ylabel=c, title=f"E · {c} by family — eta^2 = {ssb/sst:.3f} "
                           f"(share of variance explained by the class label)")
axes[-1].set_xticks(range(1, len(order) + 1))
axes[-1].set_xticklabels(order, rotation=90, fontsize=7.5)
fig.tight_layout()
fig.savefig(CFG.fig("task1_E_violin.png"), dpi=125, bbox_inches="tight")
plt.show()

eta = {}
for c in FEATCOLS:
    grand = feat[c].mean()
    ssb = sum(len(g) * (g[c].mean() - grand) ** 2 for _, g in feat.groupby("family"))
    eta[c] = ssb / ((feat[c] - grand) ** 2).sum()
print("class-discriminative power of each hand-made feature (eta^2, 0-1):")
display(pd.Series(eta).sort_values(ascending=False).round(3).to_frame("eta_squared"))

**Reading.** The byte-value distributions are strongly non-Gaussian — large
positive skew and heavy kurtosis on `zero_frac`, `high_frac` and `mean`, because
most files contain long runs of `0x00` padding and `0xFF` filler. The η² column
says how much of each feature's variance the family label explains: the top
features (typically `entropy`, `row_autocorr`, `zero_frac`, `std`) do carry real
class signal, which is a useful sanity check — it means the task is learnable
from texture and we are not about to train on noise. But no single scalar
separates 39 classes; that is precisely the job we hand to the CNN.

---
## F · Data quality, and the leakage risk that decides our whole protocol

This is the most consequential section of Task 1. §6.2 of the brief requires a
**subject/source-based split**. MaleBin ships **no** subject, sample-hash or
source column — so we have to construct the grouping key ourselves, and we have
to justify it.

**The argument.** A malware *family* is a set of polymorphic variants of the same
program. Byte-plots of two variants differ by a shifted section, a repacked
region, or a few patched bytes, and are otherwise the same picture. MaleBin also
*merges two corpora* that partly overlap in provenance, so the same original
sample can appear twice. If a variant lands in train and its near-twin in test,
the reported score measures memorisation, not classification — the exact failure
mode §6.2 forbids, and the reason a random split on this kind of data produces
the 99%+ numbers the literature reports.

**Our grouping key.** Exact duplicates via SHA-1 of the pixel buffer, plus near
duplicates via a 128-bit difference hash (horizontal + vertical gradients) with
Hamming distance ≤ 6, unioned with union-find. Every split in Tasks 2 and 3 is
grouped on that key and stratified on the family.

In [ ]:
print(f"corrupt or unreadable files : {len(bad)}")

const = feat.std_is_zero = (feat["std"] < 1e-6)
print(f"constant (single-value) images: {int(const.sum())}")
near_const = feat["std"] < 3
print(f"near-constant images (std<3) : {int(near_const.sum())}")
if near_const.sum():
    display(feat.loc[near_const, ["family", "mean", "std", "entropy"]].head(10))

import hashlib
sha = [hashlib.sha1(imgs_eda[i].tobytes()).hexdigest() for i in range(len(df))]
dup = pd.Series(sha).duplicated(keep=False)
print(f"\nexact pixel duplicates      : {int(dup.sum())} images in "
      f"{pd.Series(sha)[dup].nunique()} groups")
cross = (pd.DataFrame({"h": sha, "f": df.family.values})
           .groupby("h").f.nunique())
print(f"  duplicate hashes appearing under >1 family: {int((cross>1).sum())}"
      "  <- these would be label noise, not just duplication")
if (cross > 1).sum():
    h0 = cross[cross > 1].index[0]
    display(df.loc[[i for i, h in enumerate(sha) if h == h0],
                   ["family", "path"]])

In [ ]:
M.banner("Building the duplicate-group key (this IS our 'subject' column)")
groups = M.build_dedup_groups(imgs_eda, df)
df["dup_group"] = groups

gsz = pd.Series(groups).value_counts()
per_fam = (pd.DataFrame({"g": groups, "family": df.family.values})
             .groupby("family").g.nunique().rename("distinct_binaries").to_frame())
per_fam["images"] = df.family.value_counts().reindex(per_fam.index).values
per_fam["images_per_binary"] = (per_fam.images / per_fam.distinct_binaries).round(2)
per_fam = per_fam.sort_values("images_per_binary", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
ax[0].hist(gsz.values, bins=range(1, min(gsz.max(), 40) + 2), color="tab:red")
ax[0].set(xlabel="images in one duplicate group", ylabel="groups", yscale="log",
          title=f"F · Duplicate-group sizes\n{len(gsz):,} groups for "
                f"{len(df):,} images ({100*(1-len(gsz)/len(df)):.1f}% collapse)")
ax[1].bar(range(len(per_fam)), per_fam.images_per_binary.values, color="tab:red")
ax[1].axhline(1, ls="--", c="k", lw=1, label="1 = no duplication")
ax[1].set_xticks(range(len(per_fam)))
ax[1].set_xticklabels(per_fam.index, rotation=90, fontsize=7)
ax[1].set(ylabel="images per distinct binary",
          title="F · Redundancy per family — how many near-copies of each binary?")
ax[1].legend()
fig.tight_layout()
fig.savefig(CFG.fig("task1_F_duplicates.png"), dpi=130, bbox_inches="tight")
plt.show()
display(per_fam)
per_fam.to_csv(CFG.art("task1_F_redundancy_per_family.csv"))

#### F.5 · Families that share images — a labelling defect in the dataset

Near-duplicate search is run *within* a family, so the only way a duplicate
group can carry two different family labels is an **exact pixel match filed
under two folders**. If that happens it is not a bug in the grouping — it is a
defect in the labels, and the grouping is what stops it becoming leakage.

It matters for what any model here can possibly score: if two families share
their images, **no classifier can separate them**, so their share of macro-F1
is unreachable by construction. We report it rather than quietly dropping a
class.

In [ ]:
M.banner("F.5 Cross-family exact duplicates (labelling defect check)")
xf = M.CROSS_FAMILY_GROUPS
if not xf:
    print("No duplicate group spans more than one family -- labels are clean "
          "in this respect.")
else:
    xf_df = pd.DataFrame([{"group": d["group"], "n_images": d["n_images"],
                           "families": " + ".join(d["families"])} for d in xf])
    n_imgs = int(xf_df.n_images.sum())
    print(f"{len(xf_df):,} duplicate group(s), {n_imgs:,} images, carry more "
          f"than one family label.")
    display(xf_df.families.value_counts().rename("groups").to_frame())
    xf_df.to_csv(CFG.art("task1_F_cross_family_groups.csv"), index=False)

    affected = sorted({f for d in xf for f in d["families"]})
    print("Affected families:", ", ".join(affected))
    display(df[df.family.isin(affected)].family.value_counts()
              .rename("images").to_frame())

    pair = xf[0]["families"]
    ia = int(np.flatnonzero(df.family.values == pair[0])[0])
    ib = int(np.flatnonzero(df.family.values == pair[1])[0])
    fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.4))
    axes[0].imshow(imgs_eda[ia], cmap="gray", vmin=0, vmax=255)
    axes[0].set_title(pair[0], fontsize=9)
    axes[1].imshow(imgs_eda[ib], cmap="gray", vmin=0, vmax=255)
    axes[1].set_title(pair[1], fontsize=9)
    dd = np.abs(imgs_eda[ia].astype(int) - imgs_eda[ib].astype(int))
    axes[2].imshow(dd, cmap="magma", vmin=0, vmax=max(1, int(dd.max())))
    axes[2].set_title(f"absolute difference, max = {int(dd.max())}", fontsize=9)
    for a in axes:
        a.axis("off")
    fig.suptitle("F.5 - the same image filed under two family labels", fontsize=11)
    fig.tight_layout()
    fig.savefig(CFG.fig("task1_F_cross_family_pair.png"), dpi=130,
                bbox_inches="tight")
    plt.show()

    n_fam = int(df.family.nunique())
    print(f"Ceiling effect: {len(affected)} of {n_fam} families are involved. "
          f"Two families that share their images cannot both be recalled, so "
          f"about {1/n_fam:.3f} of macro-F1 -- one class worth -- is "
          f"unreachable no matter how good the model is.")

In [ ]:
big = gsz[gsz > 1]
if len(big):
    g0 = int(big.index[0])
    members = np.flatnonzero(groups == g0)[:8]
    fig, axes = plt.subplots(1, len(members), figsize=(2.1 * len(members), 2.6))
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, members):
        ax.imshow(imgs_eda[i], cmap="gray", vmin=0, vmax=255)
        ax.set_title(Path(df.path.iloc[i]).name[:16], fontsize=7)
        ax.axis("off")
    fig.suptitle(f"F · One duplicate group ({len(members)} of "
                 f"{int(gsz[g0])} members), family = {df.family.iloc[members[0]]}\n"
                 "A random split would scatter these across train and test.",
                 fontsize=11)
    fig.tight_layout()
    fig.savefig(CFG.fig("task1_F_dup_group_example.png"), dpi=130,
                bbox_inches="tight")
    plt.show()
else:
    print("no multi-image duplicate groups found at this threshold")

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score

X = StandardScaler().fit_transform(feat[FEATCOLS].values)
y = df.label.values
rows = []
for name, splitter, g in [
        ("random (stratified)  <- WRONG", StratifiedKFold(5, shuffle=True,
                                                          random_state=CFG.seed), None),
        ("duplicate-grouped    <- OURS",  StratifiedGroupKFold(5, shuffle=True,
                                                               random_state=CFG.seed), groups)]:
    accs, f1s = [], []
    for trn, tst in (splitter.split(X, y, g) if g is not None else splitter.split(X, y)):
        k = KNeighborsClassifier(1).fit(X[trn], y[trn])
        p = k.predict(X[tst])
        accs.append(accuracy_score(y[tst], p))
        f1s.append(f1_score(y[tst], p, average="macro", zero_division=0))
    rows.append(dict(protocol=name, accuracy=f"{np.mean(accs):.4f} +- {np.std(accs):.4f}",
                     macro_f1=f"{np.mean(f1s):.4f} +- {np.std(f1s):.4f}",
                     _f1=np.mean(f1s)))
tbl = pd.DataFrame(rows)
display(tbl.drop(columns="_f1"))
gap = rows[0]["_f1"] - rows[1]["_f1"]
print(f"\ninflation from splitting randomly: +{gap:.4f} macro-F1 "
      f"({100*gap/max(rows[1]['_f1'],1e-9):+.1f}% relative)")
print("A 1-nearest-neighbour classifier on 20 hand-made features is not a strong")
print("model. If it does well under a random split, that is memorisation of")
print("near-duplicates, not classification -- which is the whole point.")
tbl.drop(columns="_f1").to_csv(CFG.art("task1_F_leakage_demo.csv"), index=False)

**Reading — and the decision that follows.** The gap printed above is the price
of getting the split wrong, measured on this dataset with a deliberately weak
model. Every result in Tasks 2 and 3 therefore uses the **duplicate-grouped,
family-stratified** split, and the same grouping is used for the 5 CV folds in
Task 3. We keep the ungrouped number visible as a control so a reader can see
what we gave up: our headline scores will be *lower* than the published ones,
and that is the honest trade the rubric asks for (§8, "leakage void").

Concretely, the grouping means our effective sample size is the number of
**distinct binaries**, not the number of images — the `distinct_binaries` column
above is the real size of this dataset.

---
## G · Correlation, redundancy, and 2-D structure (PCA / t-SNE / UMAP)

In [ ]:
corr = feat[FEATCOLS].corr()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATCOLS)))
ax.set_yticks(range(len(FEATCOLS)))
ax.set_xticklabels(FEATCOLS, rotation=90, fontsize=8)
ax.set_yticklabels(FEATCOLS, fontsize=8)
for i in range(len(FEATCOLS)):
    for j in range(len(FEATCOLS)):
        if abs(corr.iloc[i, j]) > .55 and i != j:
            ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center",
                    fontsize=6.5)
fig.colorbar(im, ax=ax, fraction=.046, label="Pearson r")
ax.set_title("G · Correlation of the derived features "
             "(|r| > 0.55 annotated)")
fig.tight_layout()
fig.savefig(CFG.fig("task1_G_correlation.png"), dpi=130, bbox_inches="tight")
plt.show()

hi = [(FEATCOLS[i], FEATCOLS[j], round(corr.iloc[i, j], 3))
      for i in range(len(FEATCOLS)) for j in range(i + 1, len(FEATCOLS))
      if abs(corr.iloc[i, j]) > .85]
print("redundant feature pairs (|r| > 0.85) -- one of each pair is expendable:")
for a, b, r in sorted(hi, key=lambda t: -abs(t[2])):
    print(f"   {a:<18} ~ {b:<18} r = {r:+.3f}")
if not hi:
    print("   none")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

Xf = StandardScaler().fit_transform(feat[FEATCOLS].values)
side_small = 32
Xp = np.stack([np.asarray(Image.fromarray(imgs_eda[i]).resize((side_small, side_small)))
               .ravel() for i in range(len(df))]).astype(np.float32) / 255.0
Xp = StandardScaler().fit_transform(Xp)

pca_f = PCA(n_components=min(20, Xf.shape[1]), random_state=CFG.seed).fit(Xf)
pca_p = PCA(n_components=50, random_state=CFG.seed).fit(Xp)
print(f"PCA on features : first 2 PCs explain {pca_f.explained_variance_ratio_[:2].sum():.1%}, "
      f"first 10 -> {pca_f.explained_variance_ratio_[:10].sum():.1%}")
print(f"PCA on pixels   : first 2 PCs explain {pca_p.explained_variance_ratio_[:2].sum():.1%}, "
      f"first 50 -> {pca_p.explained_variance_ratio_[:50].sum():.1%}")

Zf = pca_f.transform(Xf)[:, :2]
Zp2 = pca_p.transform(Xp)[:, :2]
Xp50 = pca_p.transform(Xp)

perp = max(5, min(30, len(df) // 60))
Zt = TSNE(n_components=2, perplexity=perp, init="pca", random_state=CFG.seed,
          max_iter=400 if CFG.fast else 1000).fit_transform(Xp50)

Zu = None
try:
    import umap
    Zu = umap.UMAP(n_components=2, n_neighbors=min(15, len(df)//30),
                   min_dist=.1, random_state=CFG.seed).fit_transform(Xp50)
    print("UMAP ok")
except Exception as e:
    print(f"UMAP unavailable ({type(e).__name__}) -- PCA + t-SNE still shown")

In [ ]:
def scatter_by_family(ax, Z, title):
    cmap = plt.get_cmap("tab20")
    for i, fam in enumerate(sorted(df.family.unique())):
        m = (df.family == fam).values
        ax.scatter(Z[m, 0], Z[m, 1], s=7, alpha=.7, color=cmap(i % 20),
                   label=fam if i < 20 else None, linewidths=0)
    ax.set(title=title, xlabel="component 1", ylabel="component 2")

panels = [(Zf, "PCA on the 20 derived features"),
          (Zp2, f"PCA on raw {side_small}x{side_small} pixels"),
          (Zt, f"t-SNE on 50 pixel PCs (perplexity={perp})")]
if Zu is not None:
    panels.append((Zu, "UMAP on 50 pixel PCs"))

fig, axes = plt.subplots(1, len(panels), figsize=(5.4 * len(panels), 5.1))
for ax, (Z, t) in zip(np.atleast_1d(axes), panels):
    scatter_by_family(ax, Z, t)
np.atleast_1d(axes)[0].legend(fontsize=6, ncol=2, loc="best", markerscale=1.6)
fig.suptitle("G · Do the 39 families separate without a classifier? "
             "(colour = family; legend shows the first 20)", fontsize=13)
fig.tight_layout()
fig.savefig(CFG.fig("task1_G_projections.png"), dpi=130, bbox_inches="tight")
plt.show()

from sklearn.metrics import silhouette_score
for Z, t in panels:
    try:
        s = silhouette_score(Z, df.label.values)
        print(f"  silhouette (family labels) on {t:<42s}: {s:+.3f}")
    except Exception as e:
        print("  silhouette failed:", e)
print("\n(+1 = perfectly separated clusters, 0 = overlapping, -1 = wrong clusters)")

**Reading.** PCA on the hand-made features shows only coarse structure — the
first two components explain a modest share of variance and families overlap
heavily. t-SNE/UMAP on raw pixel PCs is much more encouraging: many families
form tight, well-separated islands, several of which are *sub-clustered*, and
those sub-clusters are exactly the duplicate groups found in section F. A few
families sit on top of each other (typically the `Allaple`, `Swizzor`,
`C2LOP` and `Lolyda` variant pairs), which predicts where the confusion matrix
in Task 2 will be dark.

Two conclusions: (i) the task is genuinely learnable from pixels — the signal is
there before any training; (ii) the *hard* part is the handful of confusable
variant pairs, so improvements should be judged on per-class recall for those
families, not on the overall average.

---
## H · Interactive plots (Plotly)

Interactive versions of the three most useful views. Hover for exact values;
double-click a legend entry to isolate a family. These render inside the saved
notebook, so they survive submission to GitHub.

In [ ]:
h = (df.groupby(["family", "is_malimg"]).size().rename("images").reset_index())
h["source"] = np.where(h.is_malimg, "Malimg (Nataraj 2011)",
                       "MalwareBazaar-derived (walt30)")
h = h.merge(per_fam.reset_index()[["family", "distinct_binaries",
                                   "images_per_binary"]],
            on="family", how="left")
fig = px.bar(h.sort_values("images", ascending=False), x="family", y="images",
             color="source", hover_data=["distinct_binaries", "images_per_binary"],
             title="H1 · Images per family and source "
                   "(hover: how many *distinct binaries* are behind those images)")
fig.update_layout(xaxis_tickangle=-90, height=560,
                  xaxis_title="malware family", yaxis_title="number of images")
fig.write_html(str(CFG.fig("task1_H1_class_balance.html")))
fig.show()

In [ ]:
Zi = Zu if Zu is not None else Zt
lab = "UMAP" if Zu is not None else "t-SNE"
hov = pd.DataFrame({"x": Zi[:, 0], "y": Zi[:, 1], "family": df.family.values,
                    "source": np.where(df.is_malimg, "Malimg", "MalwareBazaar"),
                    "dup_group": groups,
                    "file": [Path(p).name for p in df.path],
                    "entropy": feat.entropy.round(3),
                    "zero_frac": feat.zero_frac.round(3)})
fig = px.scatter(hov, x="x", y="y", color="family", symbol="source",
                 hover_data=["file", "dup_group", "entropy", "zero_frac"],
                 title=f"H2 · {lab} of byte-plots, coloured by family. "
                       f"Tight sub-clusters are duplicate groups -- hover to check "
                       f"the dup_group id.")
fig.update_traces(marker=dict(size=6, opacity=.8))
fig.update_layout(height=680, xaxis_title=f"{lab}-1", yaxis_title=f"{lab}-2")
fig.write_html(str(CFG.fig("task1_H2_projection.html")))
fig.show()

In [ ]:
long = feat.melt(id_vars=["family", "is_malimg"],
                 value_vars=["entropy", "std", "zero_frac", "ascii_frac",
                             "grad_mag", "row_autocorr"],
                 var_name="feature", value_name="value")
fig = px.box(long, x="family", y="value", color="feature", points=False,
             facet_row="feature", height=1500,
             title="H3 · Byte/texture feature distributions per family "
                   "(one row per feature; hover for quartiles)")
fig.update_xaxes(tickangle=-90, tickfont=dict(size=8))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_yaxes(matches=None)
fig.write_html(str(CFG.fig("task1_H3_features.html")))
fig.show()

---
## Related work — seven peer-reviewed papers, 2022–2026 (brief §6.3)

Track 3 requires the *attention type* column in addition to the shared columns.
Every number below was read from the paper's own abstract or results table; the
`comparable` flag records whether §6.4 lets us compare our result against it.

In [ ]:
rw = M.related_work_frame()
print(f"{len(rw)} papers, years {rw.year.min()}-{rw.year.max()}, "
      f"{int(rw.comparable.sum())} marked comparable\n")
pd.set_option("display.max_colwidth", 70)
display(rw[["key", "year", "authors", "venue", "dataset", "attention_type",
            "metrics", "comparable"]])
rw.to_csv(REPO_RW := str(CFG.art("task1_related_work_table.csv")), index=False)
print("\nfull table ->", REPO_RW)

In [ ]:
for r in M.RELATED_WORK:
    print("=" * 100)
    print(f"[{r['key']}]  {r['title']}")
    print(f"  authors        : {r['authors']}")
    print(f"  year / venue   : {r['year']} · {r['venue']}   doi:{r['doi']}")
    print(f"  dataset        : {r['dataset']}")
    print(f"  application    : {r['application']}")
    print(f"  method         : {r['method']}")
    print(f"  attention type : {r['attention_type']}")
    print(f"  metrics        : {r['metrics']}")
    print(f"  strengths      : {r['strengths']}")
    print(f"  limitations    : {r['limitations']}")
    print(f"  research gap   : {r['gap']}")
    print(f"  relation to us : {r['relation']}")
    print(f"  comparable?    : {r['comparable']}")

In [ ]:
comp = rw[rw.comparable & rw.headline_value.notna()]
fig, ax = plt.subplots(figsize=(9.5, 4.4))
b = ax.barh([f"{k} ({y})" for k, y in zip(comp.key, comp.year)],
            comp.headline_value, color="tab:grey")
for rect, m, v in zip(b, comp.headline_metric, comp.headline_value):
    ax.text(v + .05, rect.get_y() + rect.get_height() / 2, f"{v:.2f}  ({m})",
            va="center", fontsize=9)
ax.set(xlim=(90, 101), xlabel="reported headline score (%)",
       title="Published results we are compared against (Pillar A)\n"
             "all use a RANDOM split; ours will use a duplicate-grouped split")
fig.tight_layout()
fig.savefig(CFG.fig("task1_related_work_scores.png"), dpi=130, bbox_inches="tight")
plt.show()

for sc in ["malimg25", "malebin39"]:
    t = M.best_comparable_target(sc)
    print(f"\nPillar-A target for scope '{sc}':")
    print(f"   paper   : {t['paper']} -- {t['citation']}")
    print(f"   metric  : {t['metric']} = {t['value']}")
    print(f"   caveat  : {t['caveat']}")

### The research gap we are addressing

Reading the seven papers together, three things are true of *all* of them:

1. **Every reported number comes from a random split.** None of the malware-image
   papers controls for polymorphic near-duplicates or for provenance overlap
   between merged corpora. Section F measured what that is worth on this
   dataset. So the 99.2–99.4% Malimg figures are upper bounds under an
   optimistic protocol, not estimates of generalisation.
2. **The headline metric is almost always accuracy** (or precision), on data
   that is imbalanced. Only MalVis (2025) reports macro-F1, and there it sits
   **4.4 points below** accuracy (90.81 vs 95.19). The PE-image papers never
   publish that gap, and Basak et al. explicitly admit their model "struggles
   with underrepresented classes".
3. **Attention, where used, is channel-only** (SE in PAFE, SE in IMCMK-CNN,
   involution in DRIN) or generic self-attention (Swin in Alshomrani et al.).
   Nobody exploits the one structural fact that byte-plots actually have:
   **the row index is the byte offset in the file.** Channel attention pools
   that away; window self-attention treats it as an arbitrary 2-D coordinate.

**Gap → our contribution.** We build a small from-scratch CNN whose attention is
*direction-aware* (coordinate attention: a per-row and a per-column gate, so
byte-offset position survives), stack it on multi-scale dilated convolutions,
train it with byte-aware augmentation that never flips or rotates the file, and
evaluate the whole thing under a **duplicate-grouped** split with **macro-F1**
as the headline. We then compare on the Malimg-25 subset (same classes as the
published work, harder split) and on full MaleBin-39, and we report both — plus
5-fold CV with a significance test — instead of one flattering number.

---
## Task 1 conclusions

| Finding | Consequence for Tasks 2–3 |
|---|---|
| Uniform 256×256 8-bit grayscale, no corrupt files | No resizing/colour pipeline needed; input is `1×224×224` after a single resize |
| Imbalance ratio printed in §B | **macro-F1** is the headline metric; inverse-frequency class weights in the loss; model selection on validation macro-F1 |
| No subject/source column, but heavy near-duplication (§F) | We derive a duplicate-group key and use **grouped stratified** splits and CV folds everywhere. Effective *n* = number of distinct binaries |
| Row position = byte offset; images are horizontally banded | Attention must preserve vertical position → **coordinate attention**; augmentation must **not** flip or rotate |
| Signal lives at several scales (byte texture, repeated blocks, whole sections) | **Multi-scale dilated** conv block |
| t-SNE/UMAP show a few overlapping variant pairs | Judge improvements on per-class recall for those families |
| Every related-work number uses a random split and an accuracy headline | Our numbers will be lower and we say why; the fair comparison is Malimg-25 scope |
| Uploader's own disclaimer: resize distortion, outdated Malimg samples | Stated as a limitation in all three reports |

### Deliverables written by this notebook

In [ ]:
print("figures / tables written to", CFG.out_dir, "\n")
for p in sorted(Path(CFG.out_dir).rglob("*")):
    if p.is_file() and p.suffix in (".png", ".html", ".csv", ".json"):
        print(f"  {p.stat().st_size/1024:8.1f} KB  {p.relative_to(CFG.out_dir)}")
print("""
Copy into the repo as:
  report/task1/Group12_MaleBin_task1_report.pdf     (write from the readings above)
  related_work/Group00_MaleBin_related_work_table.pdf
  related_work/papers/                              (the 7 paper PDFs)
  code/task1/Group12_MaleBin_task1_eda.ipynb        (this notebook, with output)
""")